In [1]:
import os

# Make only physical GPU 1 visible
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("CUDA_VISIBLE_DEVICES set to 1. Restart the kernel now.")

CUDA_VISIBLE_DEVICES set to 1. Restart the kernel now.


In [ ]:
import torch, os
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count:", torch.cuda.device_count())
print("gpu0 name:", torch.cuda.get_device_name(0))

In [ ]:
import os, json, random
from typing import Optional, Dict, List, Any, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm import HyperbolicLCM


class EvalConfig:
    subject_name = "anatomy"

    cache_dir = "mcq_cache"
    out_dir = "runs/mcq_hlcm_mmlu_auxtrain"

    encoder_name = "microsoft/deberta-v3-small"
    chunk_tok_len = 256
    seq_len = 8
    encoder_batch_size = 64

    hlcm_ckpt_path = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    mcq_ckpt_path = "runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt"
    normalizer_path = "normalizer.pt"

    in_dim = 768
    model_dim = 4096
    num_heads = 32
    num_layers = 12
    ffn_mult = 4
    dropout = 0.30
    manifold_c = 0.002
    causal = True
    input_scale = 0.05
    input_max_norm = 1.0

    eval_batch_size = 8
    num_workers = 0   # safer for notebooks
    prefer_gpu_index = 0
    seed = 42
    head_dropout = 0.50


cfg = EvalConfig()


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index=0):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    idx = min(prefer_gpu_index, torch.cuda.device_count() - 1)
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def answerkey_to_index(answer_key, num_choices):
    if isinstance(answer_key, int):
        idx = int(answer_key)
    else:
        answer_key = str(answer_key).strip()
        alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
        numeric = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4}

        if answer_key in alpha:
            idx = alpha[answer_key]
        elif answer_key in numeric:
            idx = numeric[answer_key]
        else:
            raise ValueError(f"Unknown answer label: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"Answer index {idx} >= number of choices {num_choices}")

    return idx


def load_normalizer(normalizer_path, device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def resolve_existing_split(ds_dict, preferred_name, fallback_names):
    if preferred_name in ds_dict:
        return preferred_name

    for name in fallback_names:
        if name in ds_dict:
            return name

    raise KeyError(f"Could not find split {preferred_name}. Available: {list(ds_dict.keys())}")


def load_mmlu_subject(subject_name):
    return load_dataset("cais/mmlu", subject_name)


def unwrap_mmlu_row(example):
    if isinstance(example, dict) and len(example) == 1:
        only_key = next(iter(example.keys()))
        only_val = example[only_key]
        if only_key in {"train", "validation", "val", "test", "dev"} and isinstance(only_val, dict):
            return only_val
    return example


def normalize_choices_field(example):
    example = unwrap_mmlu_row(example)

    if "question" in example:
        stem = str(example["question"])
    elif "input" in example:
        stem = str(example["input"])
    elif "prompt" in example:
        stem = str(example["prompt"])
    else:
        raise KeyError(f"Question field not found. keys={list(example.keys())}")

    if "choices" in example:
        raw_choices = example["choices"]
        if isinstance(raw_choices, list):
            choice_texts = [str(x) for x in raw_choices]
        elif isinstance(raw_choices, dict) and "text" in raw_choices:
            choice_texts = [str(x) for x in raw_choices["text"]]
        else:
            raise TypeError(f"Unsupported choices format: {type(raw_choices)}")
    elif all(k in example for k in ["A", "B", "C", "D"]):
        choice_texts = [str(example["A"]), str(example["B"]), str(example["C"]), str(example["D"])]
        if "E" in example:
            choice_texts.append(str(example["E"]))
    else:
        raise KeyError(f"Choice field not found. keys={list(example.keys())}")

    if "answer" in example:
        answer_value = example["answer"]
    elif "answerKey" in example:
        answer_value = example["answerKey"]
    elif "target" in example:
        answer_value = example["target"]
    elif "label" in example and not isinstance(example["label"], list):
        answer_value = example["label"]
    else:
        raise KeyError(f"Answer field not found. keys={list(example.keys())}")

    label = answerkey_to_index(answer_value, len(choice_texts))
    return stem, choice_texts, label


class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts):
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text):
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text):
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            out.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem, choice_texts):
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


def build_or_load_cached_split(cache_name_prefix, split_name, split_data, conceptizer, cache_dir):
    ensure_dir(cache_dir)

    safe_ds = cache_name_prefix.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt",
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0

    print(f"[cache] building {cache_name_prefix} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cache_name_prefix}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path}; examples={len(rows)}, skipped={skipped}")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


def build_hlcm_from_cfg(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg, device):
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.hlcm_ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.hlcm_ckpt_path}")

    obj = torch.load(cfg.hlcm_ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.hlcm_ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


class MCQHead(nn.Module):
    def __init__(self, model, dropout=0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def masked_choice_cross_entropy(logits, labels, choice_mask):
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold = int(labels[i].item())
        local_pos = (valid_idx == gold).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            continue

        local_pos = int(local_pos.item())
        losses.append(-lp[local_pos])

    if len(losses) == 0:
        return torch.tensor(0.0, device=logits.device)

    return torch.stack(losses).mean()


def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)

    sq_error = (probs - one_hot) ** 2
    sq_error = sq_error.masked_fill(~choice_mask, 0.0)

    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    if confidences.numel() == 0:
        return 0.0, 0.0

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            weight = mask.float().mean().item()

            ece += weight * gap
            mce = max(mce, gap)

    return float(ece), float(mce)

@torch.no_grad()
def evaluate_ranking_metrics(model, loader, device, mu=None, sigma=None, k_values=(1, 2, 3)):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0
    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(logits, labels, choice_mask)

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total += batch_size
        correct += int(batch_correct.sum().item())

        brier_per_example = multiclass_brier_score(
            probs=probs,
            labels=labels,
            choice_mask=choice_mask,
        )
        total_brier += float(brier_per_example.sum().item())

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences_all = torch.cat(all_confidences, dim=0) if all_confidences else torch.empty(0)
    correctness_all = torch.cat(all_correctness, dim=0) if all_correctness else torch.empty(0)

    ece, mce = expected_calibration_error(
        confidences=confidences_all,
        correctness=correctness_all,
        n_bins=15,
    )

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


set_seed(cfg.seed)
device = pick_device(cfg.prefer_gpu_index)

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("Device:", device)

if not os.path.exists(cfg.mcq_ckpt_path):
    raise FileNotFoundError(f"MCQ checkpoint not found: {cfg.mcq_ckpt_path}")

conceptizer = DebertaConceptizer(
    model_name=cfg.encoder_name,
    chunk_tok_len=cfg.chunk_tok_len,
    seq_len=cfg.seq_len,
    batch_size=cfg.encoder_batch_size,
    device=device,
)

subject_ds = load_mmlu_subject(cfg.subject_name)

val_split = resolve_existing_split(subject_ds, "validation", ["validation", "val", "dev"])
test_split = resolve_existing_split(subject_ds, "test", ["test"])

val_rows = build_or_load_cached_split(
    f"mmlu_{cfg.subject_name}",
    val_split,
    subject_ds[val_split],
    conceptizer,
    cfg.cache_dir,
)

test_rows = build_or_load_cached_split(
    f"mmlu_{cfg.subject_name}",
    test_split,
    subject_ds[test_split],
    conceptizer,
    cfg.cache_dir,
)

val_loader = DataLoader(
    MCQFeatureDataset(val_rows),
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    collate_fn=mcq_collate,
)

test_loader = DataLoader(
    MCQFeatureDataset(test_rows),
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    collate_fn=mcq_collate,
)

hlcm = load_pretrained_hlcm(cfg, device)
model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

ckpt = torch.load(cfg.mcq_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"], strict=True)

mu, sigma = load_normalizer(cfg.normalizer_path, device)

val_metrics = evaluate_ranking_metrics(model, val_loader, device, mu, sigma, k_values=(1, 2, 3))
test_metrics = evaluate_ranking_metrics(model, test_loader, device, mu, sigma, k_values=(1, 2, 3))

summary = {
    "subject": cfg.subject_name,
    "checkpoint": cfg.mcq_ckpt_path,
    "validation": val_metrics,
    "test": test_metrics,
}

print(json.dumps(summary, indent=2))

save_dir = os.path.join(cfg.out_dir, cfg.subject_name)
ensure_dir(save_dir)

save_path = os.path.join(save_dir, "eval_precision_recall_ranking.json")
with open(save_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved metrics to:", save_path)

Device: cuda:0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/mmlu_anatomy_validation_seq8_tok256.pt
[cache] loading mcq_cache/mmlu_anatomy_test_seq8_tok256.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


Evaluating: 100%|███████████████████████████████████████████████████| 17/17 [00:01<00:00, 11.82it/s]

{
  "subject": "anatomy",
  "checkpoint": "runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt",
  "validation": {
    "loss": 1.384993008204869,
    "accuracy": 0.2857142857142857,
    "brier_score": 0.7493616512843541,
    "ece": 0.030898720026016235,
    "mce": 0.030898720026016235,
    "ece_bins": 15,
    "precision@1": 0.2857142857142857,
    "recall@1": 0.2857142857142857,
    "precision@2": 0.2857142857142857,
    "recall@2": 0.5714285714285714,
    "precision@3": 0.3095238095238095,
    "recall@3": 0.9285714285714286,
    "mrr": 0.5654761904761905
  },
  "test": {
    "loss": 1.3864892844800596,
    "accuracy": 0.28888888888888886,
    "brier_score": 0.7500904083251954,
    "ece": 0.034852296113967896,
    "mce": 0.034852296113967896,
    "ece_bins": 15,
    "precision@1": 0.28888888888888886,
    "recall@1": 0.28888888888888886,
    "precision@2": 0.2518518518518518,
    "recall@2": 0.5037037037037037,
    "precision@3": 0.24938271604938245,
    "recall@3": 0.7481481481481481,
   

In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# FAST DEBUG CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("anatomy",)

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_mmlu_auxtrain"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 128

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 1
    train_batch_size: int = 16
    eval_batch_size: int = 16
    grad_accum_steps: int = 1

    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True
    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 2
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    aux_train_config_name: str = "auxiliary_train"
    aux_train_split_name: str = "train"

    subject_val_split_name: str = "validation"
    subject_test_split_name: str = "test"

    # FAST DEBUG LIMITS
    debug_train_limit: int = 1000
    debug_val_limit: int = 100
    debug_test_limit: int = 200


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: Any, num_choices: int) -> int:
    if isinstance(answer_key, int):
        idx = int(answer_key)
    else:
        answer_key = str(answer_key).strip()
        alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
        numeric = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4}
        if answer_key in alpha:
            idx = alpha[answer_key]
        elif answer_key in numeric:
            idx = numeric[answer_key]
        else:
            raise ValueError(f"Unknown answer label: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answer={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded {normalizer_path}")
    return mu, sigma


def resolve_existing_split(ds_dict, preferred_name: str, fallback_names: List[str]) -> str:
    if preferred_name in ds_dict:
        return preferred_name
    for name in fallback_names:
        if name in ds_dict:
            return name
    raise KeyError(f"Could not find split '{preferred_name}'. Available: {list(ds_dict.keys())}")


# ============================================================
# LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices")

        local_pos = int(local_pos.item())
        cv = int(lp.size(0))

        if label_smoothing > 0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += 1.0 - label_smoothing
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_texts.append(
                self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            )
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            out.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_mmlu_auxiliary_train():
    return load_dataset("cais/mmlu", "auxiliary_train")


def load_mmlu_subject(subject_name: str):
    return load_dataset("cais/mmlu", subject_name)


def unwrap_mmlu_row(example: Dict[str, Any]) -> Dict[str, Any]:
    if isinstance(example, dict) and len(example) == 1:
        only_key = next(iter(example.keys()))
        only_val = example[only_key]
        if only_key in {"train", "validation", "val", "test", "dev"} and isinstance(only_val, dict):
            return only_val
    return example


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    example = unwrap_mmlu_row(example)

    if "question" in example:
        stem = str(example["question"])
    elif "input" in example:
        stem = str(example["input"])
    elif "prompt" in example:
        stem = str(example["prompt"])
    else:
        raise KeyError(f"question field not found. keys={list(example.keys())}")

    if "choices" in example:
        raw_choices = example["choices"]
        if isinstance(raw_choices, list):
            choice_texts = [str(x) for x in raw_choices]
        elif isinstance(raw_choices, dict):
            if "text" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["text"]]
            elif "label" in raw_choices:
                choice_texts = [str(x) for x in raw_choices["label"]]
            else:
                raise KeyError(f"unsupported choices dict: {raw_choices.keys()}")
        else:
            raise TypeError(f"Unsupported choices type: {type(raw_choices)}")
    elif all(k in example for k in ["A", "B", "C", "D"]):
        choice_texts = [str(example["A"]), str(example["B"]), str(example["C"]), str(example["D"])]
        if "E" in example:
            choice_texts.append(str(example["E"]))
    elif "options" in example:
        choice_texts = [str(x) for x in example["options"]]
    else:
        raise KeyError(f"choice field not found. keys={list(example.keys())}")

    if "answer" in example:
        answer_value = example["answer"]
    elif "answerKey" in example:
        answer_value = example["answerKey"]
    elif "target" in example:
        answer_value = example["target"]
    elif "label" in example and not isinstance(example["label"], list):
        answer_value = example["label"]
    else:
        raise KeyError(f"answer field not found. keys={list(example.keys())}")

    label = answerkey_to_index(answer_value, len(choice_texts))
    return stem, choice_texts, label


def build_or_load_cached_split(
    cache_name_prefix: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = cache_name_prefix.replace("/", "_")

    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    rows = []
    skipped = 0

    print(f"[cache] building {cache_name_prefix} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cache_name_prefix}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {"x": x, "choice_mask": choice_mask, "labels": labels}


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device):
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded pretrained HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# TRAIN / EVAL / INFERENCE
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device, mu, sigma):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(logits, labels, choice_mask, label_smoothing=0.0)

        preds = logits.argmax(dim=-1)

        total_loss += float(loss.item()) * x.size(0)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


@torch.no_grad()
def run_inference_with_time(model, loader, device, mu, sigma, save_path=None):
    model.eval()

    total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        batch_correct = preds.eq(labels)

        for i in range(x.size(0)):
            predictions.append({
                "example_index": total + i,
                "gold": int(labels[i].item()),
                "pred": int(preds[i].item()),
                "correct": int(batch_correct[i].item()),
            })

        correct += int(batch_correct.sum().item())
        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "correct": correct,
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN FAST DEBUG TRAIN + INFERENCE
# ============================================================

def fast_debug_train_and_infer(cfg: FinetuneConfig):
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    subject_name = cfg.datasets_to_run[0]
    subject_safe = subject_name.replace("/", "_")
    out_dir = os.path.join(cfg.out_dir, subject_safe)

    ensure_dir(cfg.out_dir)
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    print(f"[device] {device}")
    print(f"[subject] {subject_name}")
    print(f"[out_dir] {out_dir}")

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    aux_ds = load_mmlu_auxiliary_train()
    aux_train_split = resolve_existing_split(
        aux_ds,
        cfg.aux_train_split_name,
        ["train", "auxiliary_train"]
    )

    subject_ds = load_mmlu_subject(subject_name)
    val_split = resolve_existing_split(
        subject_ds,
        cfg.subject_val_split_name,
        ["validation", "val", "dev"]
    )
    test_split = resolve_existing_split(
        subject_ds,
        cfg.subject_test_split_name,
        ["test"]
    )

    train_rows = build_or_load_cached_split(
        f"mmlu_auxtrain_for_{subject_name}",
        aux_train_split,
        aux_ds[aux_train_split],
        conceptizer,
        cfg.cache_dir,
    )

    val_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        val_split,
        subject_ds[val_split],
        conceptizer,
        cfg.cache_dir,
    )

    test_rows = build_or_load_cached_split(
        f"mmlu_{subject_name}",
        test_split,
        subject_ds[test_split],
        conceptizer,
        cfg.cache_dir,
    )

    # FAST DEBUG SUBSET
    train_rows = train_rows[:cfg.debug_train_limit]
    val_rows = val_rows[:cfg.debug_val_limit]
    test_rows = test_rows[:cfg.debug_test_limit]

    print(f"[debug subset] train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)
    test_ds = MCQFeatureDataset(test_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    start_train = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Fast debug train epoch {epoch}/{cfg.epochs}")

        for batch in pbar:
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits,
                        labels,
                        choice_mask,
                        label_smoothing=cfg.label_smoothing,
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits,
                    labels,
                    choice_mask,
                    label_smoothing=cfg.label_smoothing,
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])

            torch.save(
                {
                    "cfg": asdict(cfg),
                    "subject_name": subject_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                },
                best_path,
            )

            print(f"[save] best checkpoint -> {best_path}")

    train_time = time.time() - start_train

    # Reload best checkpoint before inference
    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)
    model.eval()

    inference_save_path = os.path.join(out_dir, "fast_debug_inference_results.json")

    inference_results = run_inference_with_time(
        model=model,
        loader=test_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=inference_save_path,
    )

    summary = {
        "subject_name": subject_name,
        "best_checkpoint": best_path,
        "train_time_sec": train_time,
        "train_time_hms": fmt_hms(train_time),
        "debug_train_examples": len(train_rows),
        "debug_val_examples": len(val_rows),
        "debug_test_examples": len(test_rows),
        "best_val_acc": best_val_acc,
        "test_accuracy_percent": inference_results["accuracy_percent"],
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(out_dir, "fast_debug_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== FAST DEBUG DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = FinetuneConfig(
    datasets_to_run=("anatomy",),

    # keep this path same as your pretrained HLCM checkpoint
    ckpt_path="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt",

    normalizer_path="normalizer.pt",
    out_dir="runs/mcq_hlcm_mmlu_auxtrain",
    cache_dir="mcq_cache",

    epochs=1,

    # Faster debug settings
    train_batch_size=16,
    eval_batch_size=16,
    encoder_batch_size=128,

    # Reduce these more if it is still slow
    debug_train_limit=1000,
    debug_val_limit=100,
    debug_test_limit=200,

    prefer_gpu_index=0,
    seed=42,
)

summary, inference_results = fast_debug_train_and_infer(cfg)

[device] cuda:0
[subject] anatomy
[out_dir] runs/mcq_hlcm_mmlu_auxtrain/anatomy


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/mmlu_auxtrain_for_anatomy_train_seq8_tok256.pt
[cache] loading mcq_cache/mmlu_anatomy_validation_seq8_tok256.pt
[cache] loading mcq_cache/mmlu_anatomy_test_seq8_tok256.pt
[debug subset] train=1000 val=14 test=135
[load] loaded pretrained HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded normalizer.pt


Fast debug train epoch 1/1: 100%|██████████████████████| 63/63 [00:17<00:00,  3.62it/s, loss=1.6420]


[epoch 1] train_loss=1.6420 val_loss=1.3850 val_acc=0.2857
[save] best checkpoint -> runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt


Inference: 100%|██████████████████████████████████████████████████████| 9/9 [00:02<00:00,  3.99it/s]


[save] inference results -> runs/mcq_hlcm_mmlu_auxtrain/anatomy/fast_debug_inference_results.json

==================== FAST DEBUG DONE ====================
{
  "subject_name": "anatomy",
  "best_checkpoint": "runs/mcq_hlcm_mmlu_auxtrain/anatomy/best_mcq.pt",
  "train_time_sec": 30.272852897644043,
  "train_time_hms": "00:00:30",
  "debug_train_examples": 1000,
  "debug_val_examples": 14,
  "debug_test_examples": 135,
  "best_val_acc": 0.2857142857142857,
  "test_accuracy_percent": 28.14814814814815,
  "inference_time_sec": 2.2564579052850604,
  "inference_time_hms": "00:00:02",
  "time_per_example_sec": 0.01671450300211156,
  "examples_per_second": 59.82828205383487
}
[summary saved] runs/mcq_hlcm_mmlu_auxtrain/anatomy/fast_debug_summary.json


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:

    datasets_to_run: Tuple[str, ...] = ("commonsense_qa",)

    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"

    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4

    dropout: float = 0.30
    manifold_c: float = 0.002

    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 10

    train_batch_size: int = 4
    eval_batch_size: int = 8

    grad_accum_steps: int = 1

    lr: float = 0.0
    head_lr: float = 3e-5

    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0

    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4

    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_commonsenseqa"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):

    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:

    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()

    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))

    torch.cuda.set_device(idx)

    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):

    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:

    seconds = max(float(seconds), 0.0)

    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)

    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:

    answer_key = str(answer_key).strip()

    alpha = {"A":0,"B":1,"C":2,"D":3,"E":4}

    idx = alpha[answer_key]

    if idx >= num_choices:
        raise ValueError()

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):

    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")

    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(logits, labels, choice_mask, label_smoothing=0.0):

    log_probs = F.log_softmax(logits, dim=-1)

    losses = []

    for i in range(logits.size(0)):

        valid = choice_mask[i]

        valid_idx = torch.nonzero(valid).squeeze(-1)

        lp = log_probs[i, valid]

        gold_global = labels[i].item()

        local_pos = (valid_idx == gold_global).nonzero().item()

        if label_smoothing > 0:

            target = torch.full_like(lp, label_smoothing / lp.size(0))

            target[local_pos] += (1.0 - label_smoothing)

            loss = -(target * lp).sum()

        else:

            loss = -lp[local_pos]

        losses.append(loss)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:

    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.encoder = AutoModel.from_pretrained(model_name).to(device)

        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad=False

        self.chunk_tok_len = chunk_tok_len
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.device = device

        self.embed_dim = self.encoder.config.hidden_size


    @torch.inference_mode()
    def embed(self, texts):

        inputs = self.tokenizer(texts,padding=True,truncation=True,max_length=self.chunk_tok_len,return_tensors="pt")

        inputs = {k:v.to(self.device) for k,v in inputs.items()}

        with amp.autocast(device_type="cuda",enabled=(self.device.type=="cuda"),dtype=torch.bfloat16):

            out = self.encoder(**inputs).last_hidden_state[:,0,:]

        return out.detach().cpu()


    def encode_text(self,text):

        ids = self.tokenizer(text,add_special_tokens=False)["input_ids"]

        chunks=[ids[i:i+self.chunk_tok_len] for i in range(0,len(ids),self.chunk_tok_len)]

        texts=[self.tokenizer.decode(c) for c in chunks[:self.seq_len]]

        if len(texts)==0:
            return torch.zeros(self.seq_len,self.embed_dim)

        embs=[]

        for i in range(0,len(texts),self.batch_size):

            embs.append(self.embed(texts[i:i+self.batch_size]))

        embs=torch.cat(embs)

        if embs.size(0)<self.seq_len:

            pad=torch.zeros(self.seq_len-embs.size(0),self.embed_dim)

            embs=torch.cat([embs,pad])

        return embs[:self.seq_len]


    def encode_choice_set(self,question,choices):

        feats=[]

        for c in choices:

            text=f"Question: {question}\nAnswer Choice: {c}"

            feats.append(self.encode_text(text))

        return torch.stack(feats)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(name):

    if name=="commonsense_qa":

        return load_dataset("commonsense_qa")

    raise ValueError()


def normalize_choices_field(example):

    stem=str(example["question"])

    choice_texts=list(example["choices"]["text"])

    label=answerkey_to_index(example["answerKey"],len(choice_texts))

    return stem,choice_texts,label


# ============================================================
# Dataset wrapper
# ============================================================

class MCQFeatureDataset(Dataset):

    def __init__(self,rows):
        self.rows=rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,idx):

        r=self.rows[idx]

        return {
            "x":r["x"],
            "label":torch.tensor(r["label"]),
            "choice_mask":r["choice_mask"]
        }


def mcq_collate(batch):

    max_c=max(item["x"].size(0) for item in batch)

    b=len(batch)
    t=batch[0]["x"].size(1)
    d=batch[0]["x"].size(2)

    x=torch.zeros(b,max_c,t,d)

    mask=torch.zeros(b,max_c,dtype=torch.bool)

    labels=torch.zeros(b,dtype=torch.long)

    for i,item in enumerate(batch):

        c=item["x"].size(0)

        x[i,:c]=item["x"]

        mask[i,:c]=True

        labels[i]=item["label"]

    return {"x":x,"choice_mask":mask,"labels":labels}


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):

    def __init__(self,model,dropout):

        super().__init__()

        self.model=model

        dim=model.layers[0].attn.embed_dim

        self.norm=nn.LayerNorm(dim)

        self.dropout=nn.Dropout(dropout)

        self.classifier=nn.Linear(dim,1)


    def forward(self,x,mask,mu=None,sigma=None):

        b,c,t,d=x.shape

        x=x.reshape(b*c,t,d)

        with torch.no_grad():

            h=self.model(x)

        h=h[:,-1]

        h=self.model.manifold.logmap0(h)

        h=self.norm(h)

        h=self.dropout(h)

        logits=self.classifier(h).view(b,c)

        logits=logits.masked_fill(~mask,-1e9)

        return logits


# ============================================================
# HLCM Loader
# ============================================================

def build_hlcm(cfg):

    return HyperbolicLCM(

        in_dim=cfg.in_dim,

        model_dim=cfg.model_dim,

        num_heads=cfg.num_heads,

        num_layers=cfg.num_layers,

        ffn_mult=cfg.ffn_mult,

        dropout=cfg.dropout,

        manifold_c=cfg.manifold_c,

        causal=cfg.causal,

        input_scale=cfg.input_scale,

        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg,device):

    model=build_hlcm(cfg).to(device)

    ckpt=torch.load(cfg.ckpt_path,map_location="cpu")

    state=ckpt["model"] if "model" in ckpt else ckpt

    model.load_state_dict(state,strict=False)

    model.eval()

    for p in model.parameters():
        p.requires_grad=False

    return model


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(model,loader,device,mu,sigma):

    model.eval()

    total=0
    correct=0
    total_loss=0

    for batch in loader:

        x=batch["x"].to(device)
        mask=batch["choice_mask"].to(device)
        labels=batch["labels"].to(device)

        logits=model(x,mask,mu,sigma)

        loss=masked_choice_cross_entropy(logits,labels,mask)

        total_loss+=loss.item()*x.size(0)

        preds=logits.argmax(-1)

        correct+=(preds==labels).sum().item()

        total+=x.size(0)

    model.train()

    return {"loss":total_loss/total,"acc":correct/total}


# ============================================================
# MAIN TRAIN FUNCTION
# ============================================================

def train_one_dataset(dataset_name,cfg):

    device=pick_device(cfg.prefer_gpu_index)

    set_seed(cfg.seed)

    conceptizer=DebertaConceptizer(
        cfg.encoder_name,
        cfg.chunk_tok_len,
        cfg.seq_len,
        cfg.encoder_batch_size,
        device
    )

    raw=load_raw_mcq_dataset(dataset_name)

    def build(split):

        rows=[]

        for ex in tqdm(raw[split]):

            stem,choices,label=normalize_choices_field(ex)

            x=conceptizer.encode_choice_set(stem,choices)

            rows.append({
                "x":x,
                "label":label,
                "choice_mask":torch.ones(x.size(0),dtype=torch.bool)
            })

        return rows


    train_rows=build("train")
    val_rows=build("validation")

    train_ds=MCQFeatureDataset(train_rows)
    val_ds=MCQFeatureDataset(val_rows)

    train_loader=DataLoader(train_ds,batch_size=cfg.train_batch_size,shuffle=True,collate_fn=mcq_collate)
    val_loader=DataLoader(val_ds,batch_size=cfg.eval_batch_size,collate_fn=mcq_collate)

    hlcm=load_pretrained_hlcm(cfg,device)

    model=MCQHead(hlcm,cfg.head_dropout).to(device)

    mu,sigma=load_normalizer(cfg.normalizer_path,device)

    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.head_lr)

    for epoch in range(cfg.epochs):

        model.train()

        for batch in tqdm(train_loader):

            x=batch["x"].to(device)
            mask=batch["choice_mask"].to(device)
            labels=batch["labels"].to(device)

            logits=model(x,mask,mu,sigma)

            loss=masked_choice_cross_entropy(
                logits,
                labels,
                mask,
                cfg.label_smoothing
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

        val=evaluate(model,val_loader,device,mu,sigma)

        print(f"Epoch {epoch+1} val_acc={val['acc']:.4f}")


# ============================================================
# MAIN
# ============================================================

def main():

    cfg=FinetuneConfig()

    for ds in cfg.datasets_to_run:

        train_one_dataset(ds,cfg)


if __name__=="__main__":
    main()

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:54<00:00, 10.37it/s]


Epoch 1 val_acc=0.1245


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:45<00:00, 10.82it/s]


Epoch 2 val_acc=0.1597


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:44<00:00, 10.85it/s]


...

Epoch 8 val_acc=0.2891


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:43<00:00, 10.89it/s]


Epoch 9 val_acc=0.2924


100%|████████████████████████████████████████████████████████████████████████████████████████| 2436/2436 [03:44<00:00, 10.87it/s]


Epoch 10 val_acc=0.2801


In [2]:
# EVAL ONLY: CommonsenseQA Precision@k / Recall@k / MRR
# Uses best_mcq.pt only. No training.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
from dataclasses import dataclass
from typing import Optional, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    dataset_name: str = "commonsense_qa"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_commonsenseqa"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    # pretrained HLCM checkpoint
    hlcm_ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"

    # trained MCQ checkpoint
    mcq_ckpt_path: str = "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt"

    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 8
    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 0   # safer in notebook
    prefer_gpu_index: int = 0


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0):
    if not torch.cuda.is_available():
        return torch.device("cpu")

    idx = min(prefer_gpu_index, torch.cuda.device_count() - 1)
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

    if answer_key not in alpha:
        raise ValueError(f"Unknown answer key: {answer_key}")

    idx = alpha[answer_key]

    if idx >= num_choices:
        raise ValueError(f"Answer index {idx} >= num_choices {num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    return mu, sigma


# ============================================================
# DATASET + CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device
        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def embed(self, texts):
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def encode_text(self, text):
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return torch.zeros(self.seq_len, self.embed_dim)

        chunks = [
            ids[i:i + self.chunk_tok_len]
            for i in range(0, len(ids), self.chunk_tok_len)
        ]

        texts = [
            self.tokenizer.decode(c, clean_up_tokenization_spaces=True)
            for c in chunks[:self.seq_len]
        ]

        embs = []

        for i in range(0, len(texts), self.batch_size):
            embs.append(self.embed(texts[i:i + self.batch_size]))

        embs = torch.cat(embs, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), self.embed_dim)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question, choices):
        feats = []

        for choice in choices:
            text = f"Question: {question}\nAnswer Choice: {choice}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


def normalize_commonsenseqa(example):
    question = str(example["question"])
    choices = list(example["choices"]["text"])
    label = answerkey_to_index(example["answerKey"], len(choices))
    return question, choices, label


def build_or_load_commonsenseqa_split(split_name, raw_split, conceptizer, cache_dir):
    ensure_dir(cache_dir)

    cache_path = os.path.join(
        cache_dir,
        f"commonsense_qa_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt",
    )

    if os.path.exists(cache_path):
        print(f"[cache] loading {cache_path}")
        return torch.load(cache_path)

    rows = []
    skipped = 0

    print(f"[cache] building CommonsenseQA {split_name}")

    for ex in tqdm(raw_split, desc=f"Conceptizing {split_name}"):
        try:
            question, choices, label = normalize_commonsenseqa(ex)
            x = conceptizer.encode_choice_set(question, choices)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": torch.ones(x.size(0), dtype=torch.bool),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example: {type(e).__name__}: {e}")

    torch.save(rows, cache_path)
    print(f"[cache] saved {cache_path}; examples={len(rows)}, skipped={skipped}")

    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    b = len(batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg, device):
    model = build_hlcm(cfg).to(device)

    if not os.path.exists(cfg.hlcm_ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.hlcm_ckpt_path}")

    ckpt = torch.load(cfg.hlcm_ckpt_path, map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] HLCM checkpoint: {cfg.hlcm_ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


class MCQHead(nn.Module):
    def __init__(self, model, dropout):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, d = x.shape
        x = x.reshape(b * c, t, d)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        h = h[:, -1]
        h = self.model.manifold.logmap0(h)
        h = self.norm(h)
        h = self.dropout(h)

        logits = self.classifier(h).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


# ============================================================
# LOSS + RANKING METRICS
# ============================================================

def masked_choice_cross_entropy(logits, labels, choice_mask):
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []

    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold = int(labels[i].item())
        local_pos = (valid_idx == gold).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            continue

        local_pos = int(local_pos.item())
        losses.append(-lp[local_pos])

    if len(losses) == 0:
        return torch.tensor(0.0, device=logits.device)

    return torch.stack(losses).mean()

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)

    sq_error = (probs - one_hot) ** 2
    sq_error = sq_error.masked_fill(~choice_mask, 0.0)

    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    if confidences.numel() == 0:
        return 0.0, 0.0

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            weight = mask.float().mean().item()

            ece += weight * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
    
@torch.no_grad()
def evaluate_ranking_metrics(model, loader, device, mu=None, sigma=None, k_values=(1, 2, 3, 4, 5)):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}

    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(logits, labels, choice_mask)

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total += batch_size
        correct += int(batch_correct.sum().item())

        brier_per_example = multiclass_brier_score(
            probs=probs,
            labels=labels,
            choice_mask=choice_mask,
        )
        total_brier += float(brier_per_example.sum().item())

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences_all = torch.cat(all_confidences, dim=0) if all_confidences else torch.empty(0)
    correctness_all = torch.cat(all_correctness, dim=0) if all_correctness else torch.empty(0)

    ece, mce = expected_calibration_error(
        confidences=confidences_all,
        correctness=correctness_all,
        n_bins=15,
    )

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results

# ============================================================
# RUN EVAL ONLY
# ============================================================

set_seed(cfg.seed)
device = pick_device(cfg.prefer_gpu_index)

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("Device:", device)

if not os.path.exists(cfg.mcq_ckpt_path):
    raise FileNotFoundError(
        f"MCQ checkpoint not found: {cfg.mcq_ckpt_path}\n"
        f"Change cfg.mcq_ckpt_path to your actual best_mcq.pt path."
    )

conceptizer = DebertaConceptizer(
    model_name=cfg.encoder_name,
    chunk_tok_len=cfg.chunk_tok_len,
    seq_len=cfg.seq_len,
    batch_size=cfg.encoder_batch_size,
    device=device,
)

raw = load_dataset("commonsense_qa")

val_rows = build_or_load_commonsenseqa_split(
    split_name="validation",
    raw_split=raw["validation"],
    conceptizer=conceptizer,
    cache_dir=cfg.cache_dir,
)

val_loader = DataLoader(
    MCQFeatureDataset(val_rows),
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(device.type == "cuda"),
    collate_fn=mcq_collate,
)

hlcm = load_pretrained_hlcm(cfg, device)
model = MCQHead(hlcm, cfg.head_dropout).to(device)

ckpt = torch.load(cfg.mcq_ckpt_path, map_location=device)

if "model_state" in ckpt:
    model.load_state_dict(ckpt["model_state"], strict=True)
elif "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"], strict=True)
else:
    model.load_state_dict(ckpt, strict=True)

mu, sigma = load_normalizer(cfg.normalizer_path, device)

metrics = evaluate_ranking_metrics(
    model=model,
    loader=val_loader,
    device=device,
    mu=mu,
    sigma=sigma,
    k_values=(1, 2, 3, 4, 5),
)

summary = {
    "dataset": "commonsense_qa",
    "split": "validation",
    "checkpoint": cfg.mcq_ckpt_path,
    "metrics": metrics,
}

print(json.dumps(summary, indent=2))

ensure_dir(cfg.out_dir)

save_path = os.path.join(cfg.out_dir, "eval_precision_recall_ranking.json")

with open(save_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved metrics to:", save_path)

Device: cuda:0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/commonsense_qa_validation_seq8_tok256.pt
[load] HLCM checkpoint: runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


Evaluating: 100%|█████████████████████████████████████████████████| 153/153 [00:18<00:00,  8.47it/s]


{
  "dataset": "commonsense_qa",
  "split": "validation",
  "checkpoint": "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt",
  "metrics": {
    "loss": 1.60610192648023,
    "accuracy": 0.27682227682227684,
    "brier_score": 0.7986641279700152,
    "ece": 0.07323753833770752,
    "mce": 0.07323753833770752,
    "ece_bins": 15,
    "precision@1": 0.27682227682227684,
    "recall@1": 0.27682227682227684,
    "precision@2": 0.2452907452907453,
    "recall@2": 0.4905814905814906,
    "precision@3": 0.22768222768222904,
    "recall@3": 0.683046683046683,
    "precision@4": 0.2166257166257166,
    "recall@4": 0.8665028665028665,
    "precision@5": 0.19999999999999563,
    "recall@5": 1.0,
    "mrr": 0.5204204204204214
  }
}
Saved metrics to: runs/mcq_hlcm_commonsenseqa/eval_precision_recall_ranking.json


In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "commonsense_qa"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_commonsenseqa"

    # Fine-tuned checkpoint. If missing, code still runs for timing with random MCQ head.
    best_ckpt_path: str = "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt"

    # Pretrained HLCM backbone checkpoint.
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    head_dropout: float = 0.50

    eval_batch_size: int = 8
    num_workers: int = 4
    prefer_gpu_index: int = 0
    seed: int = 42

    # CommonSenseQA test labels are usually unavailable, so validation is best for accuracy.
    split: str = "validation"

    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

    if answer_key not in alpha:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    idx = alpha[answer_key]

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} gives idx={idx}, but num_choices={num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device
        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def embed(self, texts: List[str]) -> torch.Tensor:
        if len(texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def encode_text(self, text: str) -> torch.Tensor:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunks = [ids[i:i + self.chunk_tok_len] for i in range(0, len(ids), self.chunk_tok_len)]
        texts = [
            self.tokenizer.decode(c, clean_up_tokenization_spaces=True)
            for c in chunks[:self.seq_len]
        ]

        embs = []
        for i in range(0, len(texts), self.batch_size):
            embs.append(self.embed(texts[i:i + self.batch_size]))

        embs = torch.cat(embs, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), self.embed_dim, dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question: str, choices: List[str]) -> torch.Tensor:
        feats = []
        for c in choices:
            text = f"Question: {question}\nAnswer Choice: {c}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_raw_mcq_dataset(name: str):
    if name == "commonsense_qa":
        return load_dataset("commonsense_qa")
    raise ValueError(f"Unknown dataset: {name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], Optional[int]]:
    stem = str(example["question"])
    choice_texts = list(example["choices"]["text"])

    # Test split may not contain labels.
    if "answerKey" not in example or example["answerKey"] is None or str(example["answerKey"]).strip() == "":
        label = None
    else:
        label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def cache_path_for_split(cfg: InferenceConfig, split_name: str) -> str:
    safe_ds = cfg.dataset_name.replace("/", "_")
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_seq{cfg.seq_len}_tok{cfg.chunk_tok_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    split_data,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    ensure_dir(cfg.cache_dir)
    path = cache_path_for_split(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(f"Cache not found: {path}")

    if conceptizer is None:
        raise RuntimeError("Conceptizer is required to build cache.")

    rows = []
    skipped = 0

    print(f"[cache] building {cfg.dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cfg.dataset_name}-{split_name}"):
        try:
            stem, choices, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choices)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": -1 if label is None else int(label),
                "has_label": label is not None,
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r.get("label", -1), dtype=torch.long),
            "has_label": torch.tensor(bool(r.get("has_label", r.get("label", -1) != -1)), dtype=torch.bool),
            "choice_mask": r["choice_mask"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    b = len(batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.full((b,), -1, dtype=torch.long)
    has_label = torch.zeros(b, dtype=torch.bool)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]
        has_label[i] = item["has_label"]

    return {
        "x": x,
        "choice_mask": mask,
        "labels": labels,
        "has_label": has_label,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model, dropout):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(self, x, mask, mu=None, sigma=None):
        b, c, t, d = x.shape
        x = x.reshape(b * c, t, d)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        h = h[:, -1]
        h = self.model.manifold.logmap0(h)
        h = self.norm(h)
        h = self.dropout(h)

        logits = self.classifier(h).view(b, c)
        logits = logits.masked_fill(~mask, -1e9)

        return logits


def build_hlcm(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_inference_model(cfg: InferenceConfig, device: torch.device):
    hlcm = build_hlcm(cfg).to(device)
    model = MCQHead(hlcm, cfg.head_dropout).to(device)

    if os.path.exists(cfg.best_ckpt_path):
        print(f"[load] loading fine-tuned checkpoint: {cfg.best_ckpt_path}")
        obj = torch.load(cfg.best_ckpt_path, map_location=device)

        if "model_state" not in obj:
            raise KeyError("Checkpoint exists, but does not contain 'model_state'.")

        missing, unexpected = model.load_state_dict(obj["model_state"], strict=False)

        print(f"[load] fine-tuned model loaded")
        print(f"[load] missing keys: {len(missing)}")
        print(f"[load] unexpected keys: {len(unexpected)}")

    else:
        print("=" * 80)
        print("[warning] fine-tuned best_mcq.pt not found.")
        print("[warning] using pretrained HLCM backbone + random MCQ head.")
        print("[warning] inference time is valid, but accuracy is NOT meaningful.")
        print("=" * 80)

        if os.path.exists(cfg.ckpt_path):
            ckpt = torch.load(cfg.ckpt_path, map_location="cpu")
            state = ckpt["model"] if "model" in ckpt else ckpt
            missing, unexpected = hlcm.load_state_dict(state, strict=False)

            print(f"[load] pretrained HLCM loaded from {cfg.ckpt_path}")
            print(f"[load] HLCM missing keys: {len(missing)}")
            print(f"[load] HLCM unexpected keys: {len(unexpected)}")
        else:
            print("[warning] pretrained HLCM checkpoint also not found.")
            print("[warning] using fully random HLCM + random MCQ head.")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# INFERENCE
# ============================================================

@torch.no_grad()
def run_inference_with_time(model, loader, device, mu=None, sigma=None, save_path=None):
    model.eval()

    total = 0
    labeled_total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        has_label = batch["has_label"].to(device, non_blocking=True)

        logits = model(x, mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        if has_label.any():
            labeled_total += int(has_label.sum().item())
            correct += int((preds[has_label] == labels[has_label]).sum().item())

        for i in range(x.size(0)):
            item = {
                "example_index": total + i,
                "pred": int(preds[i].item()),
                "has_label": bool(has_label[i].item()),
            }

            if bool(has_label[i].item()):
                item["gold"] = int(labels[i].item())
                item["correct"] = int(preds[i].item() == labels[i].item())

            predictions.append(item)

        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "num_labeled_examples": labeled_total,
        "correct": correct if labeled_total > 0 else None,
        "accuracy": None if labeled_total == 0 else correct / labeled_total,
        "accuracy_percent": None if labeled_total == 0 else 100.0 * correct / labeled_total,
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_commonsenseqa(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print(f"[device] {device}")

    ensure_dir(cfg.cache_dir)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    raw = load_raw_mcq_dataset(cfg.dataset_name)

    cache_path = cache_path_for_split(cfg, cfg.split)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer = DebertaConceptizer(
            cfg.encoder_name,
            cfg.chunk_tok_len,
            cfg.seq_len,
            cfg.encoder_batch_size,
            device,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        cache_start = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw[cfg.split],
            conceptizer=conceptizer,
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        cache_build_time_sec = time.perf_counter() - cache_start

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw[cfg.split],
            conceptizer=None,
        )

    print(f"[data] split={cfg.split}, examples={len(rows)}")

    ds = MCQFeatureDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    model = load_inference_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    save_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_results.json",
    )

    inference_results = run_inference_with_time(
        model=model,
        loader=loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=save_path,
    )

    summary = {
        "dataset_name": cfg.dataset_name,
        "split": cfg.split,
        "checkpoint": cfg.best_ckpt_path,
        "cache_file": cache_path,
        "num_examples": inference_results["num_examples"],
        "num_labeled_examples": inference_results["num_labeled_examples"],
        "correct": inference_results["correct"],
        "accuracy": inference_results["accuracy"],
        "accuracy_percent": inference_results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = InferenceConfig(
    dataset_name="commonsense_qa",

    best_ckpt_path="runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt",
    ckpt_path="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt",
    normalizer_path="normalizer.pt",

    cache_dir="mcq_cache",
    out_dir="runs/mcq_hlcm_commonsenseqa",

    split="validation",
    eval_batch_size=8,
    prefer_gpu_index=0,

    build_cache_if_missing=True,
)

summary, inference_results = inference_only_commonsenseqa(cfg)

[device] cuda:0


Using the latest cached version of the dataset since commonsense_qa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/user/.cache/huggingface/datasets/commonsense_qa/default/0.0.0/94630fe30dad47192a8546eb75f094926d47e155 (last modified on Mon Mar  9 15:29:08 2026).


[cache] not found: mcq_cache/commonsense_qa_validation_seq8_tok256.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building commonsense_qa / validation


Conceptizing commonsense_qa-validation: 100%|███████████████████| 1221/1221 [00:30<00:00, 40.30it/s]


[cache] saved mcq_cache/commonsense_qa_validation_seq8_tok256.pt (1221 examples, skipped=0)
[data] split=validation, examples=1221
[warning] fine-tuned best_mcq.pt not found.
[warning] using pretrained HLCM backbone + random MCQ head.
[warning] inference time is valid, but accuracy is NOT meaningful.
[load] pretrained HLCM loaded from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] HLCM missing keys: 0
[load] HLCM unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference: 100%|██████████████████████████████████████████████████| 153/153 [00:18<00:00,  8.31it/s]


[save] inference results -> runs/mcq_hlcm_commonsenseqa/commonsense_qa/inference_only_validation_results.json

==================== INFERENCE DONE ====================
{
  "dataset_name": "commonsense_qa",
  "split": "validation",
  "checkpoint": "runs/mcq_hlcm_commonsenseqa/commonsense_qa/best_mcq.pt",
  "cache_file": "mcq_cache/commonsense_qa_validation_seq8_tok256.pt",
  "num_examples": 1221,
  "num_labeled_examples": 1221,
  "correct": 246,
  "accuracy": 0.20147420147420148,
  "accuracy_percent": 20.147420147420146,
  "cache_build_time_sec": 30.442058730870485,
  "cache_build_time_hms": "00:00:30",
  "inference_time_sec": 18.485472416039556,
  "inference_time_hms": "00:00:18",
  "time_per_example_sec": 0.015139617048353446,
  "examples_per_second": 66.05186886868833
}
[summary saved] runs/mcq_hlcm_commonsenseqa/commonsense_qa/inference_only_validation_summary.json


In [10]:

# ============================================================
# ARC-Easy + ARC-Challenge
# Train + Evaluate with Accuracy, Precision@k, Recall@k, MRR
# Best checkpoint selected by validation MRR
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy", "ARC-Challenge")
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 10
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1

    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_arc_only"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()

    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []

    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")

        local_pos = int(local_pos.item())
        cv = int(lp.size(0))

        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return []

        chunk_texts = []

        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []

        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []

        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "ARC-Easy":
        return load_dataset("allenai/ai2_arc", "ARC-Easy")
    if dataset_name == "ARC-Challenge":
        return load_dataset("allenai/ai2_arc", "ARC-Challenge")

    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    stem = str(example["question"])

    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")

    choice_texts = list(choices["text"])
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)

    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt",
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0

    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": torch.ones(x.size(0), dtype=torch.bool),
                "num_choices": int(x.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)

        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)

        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# SCHEDULER
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    k_values=(1, 2, 3),
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    metric_sums = {}

    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)

        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total += batch_size

        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0

                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")

    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name=dataset_name,
        split_name="train",
        split_data=raw_ds["train"],
        conceptizer=conceptizer,
        cache_dir=cfg.cache_dir,
    )

    val_rows = build_or_load_cached_split(
        dataset_name=dataset_name,
        split_name="validation",
        split_data=raw_ds["validation"],
        conceptizer=conceptizer,
        cache_dir=cfg.cache_dir,
    )

    test_rows = build_or_load_cached_split(
        dataset_name=dataset_name,
        split_name="test",
        split_data=raw_ds["test"],
        conceptizer=conceptizer,
        cache_dir=cfg.cache_dir,
    )

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)
    test_ds = MCQFeatureDataset(test_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_score = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)

        val_metrics = evaluate_ranking_metrics(
            model=model,
            loader=val_loader,
            device=device,
            mu=mu,
            sigma=sigma,
            k_values=(1, 2, 3),
        )

        test_metrics = evaluate_ranking_metrics(
            model=model,
            loader=test_loader,
            device=device,
            mu=mu,
            sigma=sigma,
            k_values=(1, 2, 3),
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,

            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_precision@1": val_metrics["precision@1"],
            "val_recall@1": val_metrics["recall@1"],
            "val_precision@2": val_metrics["precision@2"],
            "val_recall@2": val_metrics["recall@2"],
            "val_precision@3": val_metrics["precision@3"],
            "val_recall@3": val_metrics["recall@3"],
            "val_mrr": val_metrics["mrr"],

            "test_loss": test_metrics["loss"],
            "test_acc": test_metrics["accuracy"],
            "test_precision@1": test_metrics["precision@1"],
            "test_recall@1": test_metrics["recall@1"],
            "test_precision@2": test_metrics["precision@2"],
            "test_recall@2": test_metrics["recall@2"],
            "test_precision@3": test_metrics["precision@3"],
            "test_recall@3": test_metrics["recall@3"],
            "test_mrr": test_metrics["mrr"],
        }

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_R@2={val_metrics['recall@2']:.4f} "
            f"val_R@3={val_metrics['recall@3']:.4f} "
            f"val_MRR={val_metrics['mrr']:.4f} "
            f"test_acc={test_metrics['accuracy']:.4f} "
            f"test_MRR={test_metrics['mrr']:.4f}"
        )

        if val_metrics["mrr"] > best_val_score:
            best_val_score = float(val_metrics["mrr"])

            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_metric": "mrr",
                    "best_val_score": best_val_score,
                    "best_val_acc": float(val_metrics["accuracy"]),
                    "best_val_mrr": float(val_metrics["mrr"]),
                    "history": history,
                },
                best_path,
            )

            print(f"[save] best checkpoint by validation MRR -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate_ranking_metrics(
        model=model,
        loader=val_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        k_values=(1, 2, 3),
    )

    final_test = evaluate_ranking_metrics(
        model=model,
        loader=test_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        k_values=(1, 2, 3),
    )

    summary = {
        "dataset_name": dataset_name,
        "best_checkpoint": best_path,
        "best_metric": "validation_mrr",
        "best_epoch": int(best_obj["epoch"]),

        "final_val": final_val,
        "final_test": final_test,

        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),

        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_ds),

        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_arc_only")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}

    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    save_path = os.path.join(cfg.out_dir, "all_results.json")

    with open(save_path, "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))
    print(f"Saved all results to: {save_path}")


if __name__ == "__main__":
    main()


==================== ARC-Easy ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/ARC-Easy_train_seq8_tok256.pt
[cache] loading mcq_cache/ARC-Easy_validation_seq8_tok256.pt
[cache] loading mcq_cache/ARC-Easy_test_seq8_tok256.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


Train ARC-Easy epoch 1/10: 100%|█████████████████████| 563/563 [00:45<00:00, 12.38it/s, loss=1.6011]
                                                                                                    

...

Train ARC-Easy epoch 9/10: 100%|█████████████████████| 563/563 [00:42<00:00, 13.37it/s, loss=1.5467]
                                                                                                    

[epoch 9] train_loss=1.5467 val_acc=0.2649 val_R@2=0.5088 val_R@3=0.7246 val_MRR=0.5276 test_acc=0.2588 test_MRR=0.5262


Train ARC-Easy epoch 10/10: 100%|████████████████████| 563/563 [00:42<00:00, 13.40it/s, loss=1.5593]
                                                                                                    

[epoch 10] train_loss=1.5593 val_acc=0.2667 val_R@2=0.5053 val_R@3=0.7263 val_MRR=0.5281 test_acc=0.2580 test_MRR=0.5266


OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 11.69 MiB is free. Including non-PyTorch memory, this process has 47.37 GiB memory in use. Of the allocated memory 46.97 GiB is allocated by PyTorch, and 75.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy", "ARC-Challenge")
    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_arc_only"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    # pretrained HLCM checkpoint
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 8
    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 0  # safer for notebooks
    prefer_gpu_index: int = 0


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()

    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    losses = []

    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")

        local_pos = int(local_pos.item())

        if label_smoothing > 0.0:
            cv = int(lp.size(0))
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return []

        chunk_texts = []

        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []

        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []

        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "ARC-Easy":
        return load_dataset("allenai/ai2_arc", "ARC-Easy")
    if dataset_name == "ARC-Challenge":
        return load_dataset("allenai/ai2_arc", "ARC-Challenge")

    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    stem = str(example["question"])
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")

    choice_texts = list(choices["text"])
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)

    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt",
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0

    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": torch.ones(x.size(0), dtype=torch.bool),
                "num_choices": int(x.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)

        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)

        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: EvalConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq_error = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece, mce = 0.0, 0.0

    for i in range(n_bins):
        lo, hi = i / n_bins, (i + 1) / n_bins
        mask = (confidences >= lo) & (confidences <= hi) if i == n_bins - 1 else (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
    
# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    k_values=(1, 2, 3, 4, 5),
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0
    total_brier = 0.0

    all_confidences = []
    all_correctness = []

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0
    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)

        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total += batch_size
        correct += int(batch_correct.sum().item())

        total_brier += float(
            multiclass_brier_score(probs, labels, choice_mask).sum().item()
        )

        all_confidences.append(probs.max(dim=-1).values.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences = torch.cat(all_confidences) if all_confidences else torch.empty(0)
    correctness = torch.cat(all_correctness) if all_correctness else torch.empty(0)
    ece, mce = expected_calibration_error(confidences, correctness, n_bins=15)

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


def load_mcq_checkpoint(model: MCQHead, ckpt_path: str, device: torch.device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"MCQ checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device)

    if "model_state" in ckpt:
        state = ckpt["model_state"]
    elif "state_dict" in ckpt:
        state = ckpt["state_dict"]
    else:
        state = ckpt

    model.load_state_dict(state, strict=True)

    epoch = ckpt.get("epoch", None) if isinstance(ckpt, dict) else None
    best_metric = ckpt.get("best_metric", None) if isinstance(ckpt, dict) else None
    best_val_score = ckpt.get("best_val_score", None) if isinstance(ckpt, dict) else None

    return {
        "epoch": epoch,
        "best_metric": best_metric,
        "best_val_score": best_val_score,
    }


def eval_one_dataset(dataset_name: str, cfg: EvalConfig, device: torch.device, conceptizer: DebertaConceptizer, mu, sigma):
    print(f"\n==================== EVAL ONLY: {dataset_name} ====================")

    dataset_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    best_ckpt_path = os.path.join(dataset_dir, "best_mcq.pt")

    raw_ds = load_raw_mcq_dataset(dataset_name)

    val_rows = build_or_load_cached_split(
        dataset_name=dataset_name,
        split_name="validation",
        split_data=raw_ds["validation"],
        conceptizer=conceptizer,
        cache_dir=cfg.cache_dir,
    )

    test_rows = build_or_load_cached_split(
        dataset_name=dataset_name,
        split_name="test",
        split_data=raw_ds["test"],
        conceptizer=conceptizer,
        cache_dir=cfg.cache_dir,
    )

    val_loader = DataLoader(
        MCQFeatureDataset(val_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        MCQFeatureDataset(test_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    ckpt_info = load_mcq_checkpoint(model, best_ckpt_path, device)

    val_metrics = evaluate_ranking_metrics(
        model=model,
        loader=val_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        k_values=(1, 2, 3, 4, 5),
    )

    test_metrics = evaluate_ranking_metrics(
        model=model,
        loader=test_loader,
        device=device,
        mu=mu,
        sigma=sigma,
        k_values=(1, 2, 3, 4, 5),
    )

    summary = {
        "dataset_name": dataset_name,
        "checkpoint": best_ckpt_path,
        "checkpoint_info": ckpt_info,
        "num_val_examples": len(val_rows),
        "num_test_examples": len(test_rows),
        "validation": val_metrics,
        "test": test_metrics,
    }

    ensure_dir(dataset_dir)

    save_path = os.path.join(dataset_dir, "eval_precision_recall_ranking_only.json")
    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"[saved] {save_path}")

    return summary


# ============================================================
# RUN EVAL ONLY
# ============================================================

set_seed(cfg.seed)
device = pick_device(cfg.prefer_gpu_index)

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("Device:", device)

conceptizer = DebertaConceptizer(
    model_name=cfg.encoder_name,
    chunk_tok_len=cfg.chunk_tok_len,
    seq_len=cfg.seq_len,
    batch_size=cfg.encoder_batch_size,
    device=device,
)

mu, sigma = load_normalizer(cfg.normalizer_path, device)

all_results = {}

for dataset_name in cfg.datasets_to_run:
    all_results[dataset_name] = eval_one_dataset(
        dataset_name=dataset_name,
        cfg=cfg,
        device=device,
        conceptizer=conceptizer,
        mu=mu,
        sigma=sigma,
    )

ensure_dir(cfg.out_dir)

all_save_path = os.path.join(cfg.out_dir, "arc_eval_precision_recall_ranking_only_all.json")
with open(all_save_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\n==================== ALL DONE ====================")
print(json.dumps(all_results, indent=2))
print(f"[saved] {all_save_path}")

Device: cuda:0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



==================== EVAL ONLY: ARC-Easy ====================


Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Easy' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Easy/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:57:08 2026).


[cache] loading mcq_cache/ARC-Easy_validation_seq8_tok256.pt
[cache] loading mcq_cache/ARC-Easy_test_seq8_tok256.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


{
  "dataset_name": "ARC-Easy",
  "checkpoint": "runs/mcq_hlcm_arc_only/ARC-Easy/best_mcq.pt",
  "checkpoint_info": {
    "epoch": 4,
    "best_metric": "mrr",
    "best_val_score": 0.5328947368421053
  },
  "num_val_examples": 570,
  "num_test_examples": 2376,
  "validation": {
    "loss": 1.386436562789114,
    "accuracy": 0.2719298245614035,
    "brier_score": 0.749961002876884,
    "ece": 0.018577655299445467,
    "mce": 0.6597530841827393,
    "ece_bins": 15,
    "precision@1": 0.2719298245614035,
    "recall@1": 0.2719298245614035,
    "precision@2": 0.2578947368421053,
    "recall@2": 0.5157894736842106,
    "precision@3": 0.24385964912280628,
    "recall@3": 0.7315789473684211,
    "precision@4": 0.25014619883040934,
    "recall@4": 1.0,
    "precision@5": 0.24997076023391815,
    "recall@5": 1.0,
    "mrr": 0.5328947368421053
  },
  "test": {
    "loss": 1.3855333500839644,
    "accuracy": 0.25,
    "brier_score": 0.7496909274396671,
    "ece": 0.003566904999606561,
    "mce":

Using the latest cached version of the dataset since allenai/ai2_arc couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'ARC-Challenge' at /home/user/.cache/huggingface/datasets/allenai___ai2_arc/ARC-Challenge/0.0.0/210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Mar 14 09:31:06 2026).


[cache] loading mcq_cache/ARC-Challenge_validation_seq8_tok256.pt
[cache] loading mcq_cache/ARC-Challenge_test_seq8_tok256.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


{
  "dataset_name": "ARC-Challenge",
  "checkpoint": "runs/mcq_hlcm_arc_only/ARC-Challenge/best_mcq.pt",
  "checkpoint_info": {
    "epoch": 2,
    "best_metric": null,
    "best_val_score": null
  },
  "num_val_examples": 299,
  "num_test_examples": 1172,
  "validation": {
    "loss": 1.3853263376548537,
    "accuracy": 0.20735785953177258,
    "brier_score": 0.7498809166975244,
    "ece": 0.04678575406713403,
    "mce": 0.04718555510044098,
    "ece_bins": 15,
    "precision@1": 0.20735785953177258,
    "recall@1": 0.20735785953177258,
    "precision@2": 0.23076923076923078,
    "recall@2": 0.46153846153846156,
    "precision@3": 0.23411371237458223,
    "recall@3": 0.7023411371237458,
    "precision@4": 0.25,
    "recall@4": 0.9966555183946488,
    "precision@5": 0.2506688963210703,
    "recall@5": 1.0,
    "mrr": 0.48896321070234083
  },
  "test": {
    "loss": 1.3863920766745819,
    "accuracy": 0.23976109215017063,
    "brier_score": 0.7500928148067852,
    "ece": 0.0153316784605

In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy", "ARC-Challenge")
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_arc_only"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,         # [B, C]
    labels: torch.Tensor,         # [B]
    choice_mask: torch.Tensor,    # [B, C] bool
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Cross-entropy over only valid choices.
    Label smoothing mass is distributed only over valid choices,
    never onto padded choices.
    """
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]  # [Cv]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices in row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "ARC-Easy":
        return load_dataset("allenai/ai2_arc", "ARC-Easy")
    if dataset_name == "ARC-Challenge":
        return load_dataset("allenai/ai2_arc", "ARC-Challenge")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    """
    ARC format:
      question: string
      choices: {"label": [...], "text": [...]}
      answerKey: string
    Example:
      {
        "answerKey": "B",
        "choices": {
            "label": ["A","B","C","D"],
            "text":  ["...","...","...","..."]
        },
        "id": "...",
        "question": "..."
      }
    """
    if "question" not in example:
        raise KeyError("question")
    stem = str(example["question"])

    if "choices" not in example:
        raise KeyError("choices")
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")
    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = list(choices["text"])

    if "answerKey" not in example:
        raise KeyError("answerKey")
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    """
    Frozen HLCM + LayerNorm-stabilized classifier head.
    """
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# OPTIM / SCHED
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir
    )

    val_rows = build_or_load_cached_split(
        dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir
    )

    test_rows = build_or_load_cached_split(
        dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir
    )

    if len(train_rows) == 0:
        raise RuntimeError(f"No usable training rows for {dataset_name}")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for {dataset_name}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "dataset_name": dataset_name,
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_arc_only")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))


# IMPORTANT:
# Commented out so training does NOT start again.
# if __name__ == "__main__":
#     main()

import time

@torch.inference_mode()
def infer_mcq(question, choices, best_mcq_path):
    cfg = FinetuneConfig()
    device = pick_device(cfg.prefer_gpu_index)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    ckpt = torch.load(best_mcq_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    # -----------------------------
    # START TIMER
    # -----------------------------
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    # Encode
    x = conceptizer.encode_choice_set(question, choices)
    x = x.unsqueeze(0).to(device)

    choice_mask = torch.ones(
        1,
        len(choices),
        dtype=torch.bool,
        device=device,
    )

    # Forward
    logits = model(x, choice_mask, mu=mu, sigma=sigma)
    probs = torch.softmax(logits, dim=-1).squeeze(0)

    pred_idx = int(probs.argmax().item())

    # -----------------------------
    # END TIMER
    # -----------------------------
    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    inference_time = end - start

    return {
        "answer_index": pred_idx,
        "answer": choices[pred_idx],
        "probabilities": probs.cpu().tolist(),
        "inference_time_sec": inference_time,
    }

In [2]:
result = infer_mcq(
    question="Which planet is known as the Red Planet?",
    choices=["Earth", "Mars", "Jupiter", "Venus"],
    best_mcq_path="runs/mcq_hlcm_arc_only/ARC-Easy/best_mcq.pt",
)

print(result)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
{'answer_index': 1, 'answer': 'Mars', 'probabilities': [0.248340904712677, 0.2554096579551697, 0.24843847751617432, 0.2478109896183014], 'inference_time_sec': 6.397550821304321}


In [2]:
result = infer_mcq(
    question="Which planet is known as the Red Planet?",
    choices=["Earth", "Mars", "Jupiter", "Venus"],
    best_mcq_path="runs/mcq_hlcm_arc_only/ARC-Challenge/best_mcq.pt",
)

print(result)

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
{'answer_index': 1, 'answer': 'Mars', 'probabilities': [0.249126136302948, 0.25508683919906616, 0.2478373944759369, 0.24794970452785492], 'inference_time_sec': 25.31544589996338}


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    datasets_to_run: Tuple[str, ...] = ("OpenBookQA",)
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 3
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1
    lr: float = 0.0
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 4
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_openbookqa"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")
    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


# ============================================================
# MASKED LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,         # [B, C]
    labels: torch.Tensor,         # [B]
    choice_mask: torch.Tensor,    # [B, C] bool
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Cross-entropy over valid answer choices only.
    Label smoothing is distributed only across valid choices,
    not padded choices.
    """
    b, _ = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)

    losses = []
    for i in range(b):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)  # [Cv]
        lp = log_probs[i, valid]  # [Cv]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)
        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices for row {i}")
        local_pos = int(local_pos.item())

        cv = int(lp.size(0))
        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        use_amp = (self.device.type == "cuda")
        if self.device.type == "cuda":
            dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)
        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)
        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        if len(chunk_texts) > self.seq_len:
            chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)
        t = embs.size(0)

        if t < self.seq_len:
            pad = torch.zeros(self.seq_len - t, embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


# ============================================================
# DATA PREP
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "OpenBookQA":
        return load_dataset("allenai/openbookqa")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    """
    OpenBookQA format:
      {
        'id': '7-980',
        'question_stem': 'The sun is responsible for',
        'choices': {
            'text': [...],
            'label': ['A','B','C','D']
        },
        'answerKey': 'D'
      }
    """
    if "question_stem" not in example:
        raise KeyError("question_stem")
    stem = str(example["question_stem"])

    if "choices" not in example:
        raise KeyError("choices")
    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices to be dict, got {type(choices)}")
    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = list(choices["text"])

    if "answerKey" not in example:
        raise KeyError("answerKey")
    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)
    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt"
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0
    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": choice_mask,
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    """
    Frozen HLCM + LayerNorm-stabilized classifier head.
    """
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# OPTIM / SCHED
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def evaluate(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> Dict[str, float]:
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        total_loss += float(loss.item()) * x.size(0)
        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())
        total += int(x.size(0))

    model.train()
    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


def train_one_dataset(dataset_name: str, cfg: FinetuneConfig):
    print(f"\n==================== {dataset_name} ====================")
    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(
        dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir
    )
    val_rows = build_or_load_cached_split(
        dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir
    )
    test_rows = build_or_load_cached_split(
        dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir
    )

    if len(train_rows) == 0:
        raise RuntimeError(f"No usable training rows for {dataset_name}")
    if len(val_rows) == 0:
        raise RuntimeError(f"No usable validation rows for {dataset_name}")

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = None
    if len(test_rows) > 0:
        test_ds = MCQFeatureDataset(test_rows)
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=mcq_collate,
            drop_last=False,
        )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_acc = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train {dataset_name} epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)
        val_metrics = evaluate(model, val_loader, device, mu, sigma)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }

        if test_loader is not None:
            test_metrics = evaluate(model, test_loader, device, mu, sigma)
            row["test_loss"] = test_metrics["loss"]
            row["test_acc"] = test_metrics["acc"]
        else:
            test_metrics = None

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['acc']:.4f}"
            + (f" test_acc={test_metrics['acc']:.4f}" if test_metrics is not None else "")
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = float(val_metrics["acc"])
            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "hlcm_state": model.model.state_dict(),
                    "best_val_acc": best_val_acc,
                    "history": history,
                },
                best_path,
            )
            print(f"[save] best checkpoint -> {best_path}")

    wall = time.time() - start_time

    best_obj = torch.load(best_path, map_location=device)
    model.load_state_dict(best_obj["model_state"], strict=True)

    final_val = evaluate(model, val_loader, device, mu, sigma)
    final_test = evaluate(model, test_loader, device, mu, sigma) if test_loader is not None else None

    summary = {
        "dataset_name": dataset_name,
        "best_val_acc": float(final_val["acc"]),
        "best_val_loss": float(final_val["loss"]),
        "test_acc": None if final_test is None else float(final_test["acc"]),
        "test_loss": None if final_test is None else float(final_test["loss"]),
        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),
        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_rows),
        "history": history,
    }

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_openbookqa")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--lr", type=float, default=0.0)
    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
        freeze_hlcm=True,
    )

    ensure_dir(cfg.out_dir)

    all_summaries = {}
    for ds_name in cfg.datasets_to_run:
        summary = train_one_dataset(ds_name, cfg)
        all_summaries[ds_name] = summary

    with open(os.path.join(cfg.out_dir, "all_results.json"), "w") as f:
        json.dump(all_summaries, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(json.dumps(all_summaries, indent=2))


if __name__ == "__main__":
    main()


==================== OpenBookQA ====================


/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


[cache] building OpenBookQA / train


Conceptizing OpenBookQA-train: 100%|█████████████████████████████████████████████████████████| 4957/4957 [01:38<00:00, 50.38it/s]


[cache] saved mcq_cache/OpenBookQA_train_seq8_tok256.pt (4957 examples, skipped=0)
[cache] building OpenBookQA / validation


...

[epoch 1] train_loss=1.6084 val_loss=1.3852 val_acc=0.3040 test_acc=0.3120
[save] best checkpoint -> runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt


Train OpenBookQA epoch 2/3: 100%|███████████████████████████████████████████████| 1240/1240 [01:29<00:00, 13.81it/s, loss=1.5858]


[epoch 2] train_loss=1.5858 val_loss=1.3848 val_acc=0.3100 test_acc=0.3160
[save] best checkpoint -> runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt


Train OpenBookQA epoch 3/3: 100%|███████████████████████████████████████████████| 1240/1240 [01:29<00:00, 13.83it/s, loss=1.5916]


[epoch 3] train_loss=1.5916 val_loss=1.3846 val_acc=0.3060 test_acc=0.3220
[done] OpenBookQA
{
  "dataset_name": "OpenBookQA",
  "best_val_acc": 0.31,
  "best_val_loss": 1.384789963722229,
  "test_acc": 0.316,
  "test_loss": 1.383479887008667,
  "epochs": 3,
  "wall_time_sec": 326.9669146537781,
  "wall_time_hms": "00:05:26",
  "num_train_examples": 4957,
  "num_val_examples": 500,
  "num_test_examples": 500,
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.6084150200244134,
      "val_loss": 1.3851924781799316,
      "val_acc": 0.304,
      "test_loss": 1.3838309574127197,
      "test_acc": 0.312
    },
    {
      "epoch": 2,
      "train_loss": 1.5858492537903068,
      "val_loss": 1.384789963722229,
      "val_acc": 0.31,
      "test_loss": 1.383479887008667,
      "test_acc": 0.316
    },
    {
      "epoch": 3,
      "train_loss": 1.5915510063813967,
      "val_loss": 1.3846353244781495,
      "val_acc": 0.306,
      "test_loss": 1.383381546020508,
      "test_acc": 0

In [1]:

# ============================================================
# OpenBookQA TRAIN + EVAL
# Saves small best_mcq.pt checkpoint only (NO huge HLCM save)
# Metrics: Accuracy, Precision@k, Recall@k, MRR
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import math
import json
import time
import random
import argparse
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class FinetuneConfig:
    dataset_name: str = "OpenBookQA"
    cache_dir: str = "mcq_cache"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    epochs: int = 5
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 1

    head_lr: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.10

    freeze_encoder: bool = True
    freeze_hlcm: bool = True

    head_dropout: float = 0.50

    seed: int = 42
    num_workers: int = 0  # safer for notebooks
    use_bf16: bool = True
    prefer_gpu_index: int = 0

    out_dir: str = "runs/mcq_hlcm_openbookqaa"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()

    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}")

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def move_state_to_cpu(state_dict):
    return {k: v.detach().cpu() for k, v in state_dict.items()}


# ============================================================
# LOSS
# ============================================================

def masked_choice_cross_entropy(
    logits: torch.Tensor,
    labels: torch.Tensor,
    choice_mask: torch.Tensor,
    label_smoothing: float = 0.0,
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    losses = []

    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold_global = int(labels[i].item())
        local_pos = (valid_idx == gold_global).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            raise RuntimeError(f"Gold label {gold_global} not found among valid choices for row {i}")

        local_pos = int(local_pos.item())
        cv = int(lp.size(0))

        if label_smoothing > 0.0:
            target = torch.full_like(lp, fill_value=label_smoothing / cv)
            target[local_pos] += (1.0 - label_smoothing)
            loss_i = -(target * lp).sum()
        else:
            loss_i = -lp[local_pos]

        losses.append(loss_i)

    return torch.stack(losses).mean()


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(chunk_ids, clean_up_tokenization_spaces=True)
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), embs.size(1), dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []

        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "OpenBookQA":
        return load_dataset("allenai/openbookqa")
    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    stem = str(example["question_stem"])
    choice_texts = list(example["choices"]["text"])
    label = answerkey_to_index(example["answerKey"], len(choice_texts))
    return stem, choice_texts, label


def build_or_load_cached_split(
    dataset_name: str,
    split_name: str,
    split_data,
    conceptizer: DebertaConceptizer,
    cache_dir: str,
) -> List[Dict[str, Any]]:
    ensure_dir(cache_dir)

    safe_ds = dataset_name.replace("/", "_")
    path = os.path.join(
        cache_dir,
        f"{safe_ds}_{split_name}_seq{conceptizer.seq_len}_tok{conceptizer.chunk_tok_len}.pt",
    )

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0

    print(f"[cache] building {dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)

            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": torch.ones(x.size(0), dtype=torch.bool),
                "num_choices": int(x.size(0)),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)

        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


def build_hlcm_from_cfg(cfg: FinetuneConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: FinetuneConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# SCHEDULER
# ============================================================

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_scale: float = 0.1):
        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(1, int(total_steps))
        self.min_lr_scale = float(min_lr_scale)
        self.step_num = 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]

    def step(self):
        self.step_num += 1
        s = self.step_num

        if s <= self.warmup_steps and self.warmup_steps > 0:
            scale = s / self.warmup_steps
        else:
            denom = max(1, self.total_steps - self.warmup_steps)
            p = min(1.0, max(0.0, (s - self.warmup_steps) / denom))
            cosine = 0.5 * (1.0 + math.cos(math.pi * p))
            scale = self.min_lr_scale + (1.0 - self.min_lr_scale) * cosine

        for lr0, pg in zip(self.base_lrs, self.optimizer.param_groups):
            pg["lr"] = lr0 * scale


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_ranking_metrics(model, loader, device, mu=None, sigma=None, k_values=(1, 2, 3, 4)):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0

    metric_sums["mrr"] = 0.0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)

        loss = masked_choice_cross_entropy(
            logits=logits,
            labels=labels,
            choice_mask=choice_mask,
            label_smoothing=0.0,
        )

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total += batch_size

        preds = logits.argmax(dim=-1)
        correct += int((preds == labels).sum().item())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(batch_size):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    results = {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }

    for key, value in metric_sums.items():
        results[key] = value / max(total, 1)

    return results


# ============================================================
# TRAIN
# ============================================================

def train_one_dataset(cfg: FinetuneConfig):
    dataset_name = cfg.dataset_name
    print(f"\n==================== TRAIN: {dataset_name} ====================")

    device = pick_device(cfg.prefer_gpu_index)
    set_seed(cfg.seed)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    raw_ds = load_raw_mcq_dataset(dataset_name)

    train_rows = build_or_load_cached_split(dataset_name, "train", raw_ds["train"], conceptizer, cfg.cache_dir)
    val_rows = build_or_load_cached_split(dataset_name, "validation", raw_ds["validation"], conceptizer, cfg.cache_dir)
    test_rows = build_or_load_cached_split(dataset_name, "test", raw_ds["test"], conceptizer, cfg.cache_dir)

    train_ds = MCQFeatureDataset(train_rows)
    val_ds = MCQFeatureDataset(val_rows)
    test_ds = MCQFeatureDataset(test_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    # Freeze HLCM explicitly. Train only norm + classifier head.
    for p in model.model.parameters():
        p.requires_grad = False

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        [{"params": trainable_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay}]
    )

    steps_per_epoch = math.ceil(len(train_loader) / max(cfg.grad_accum_steps, 1))
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    scheduler = WarmupCosineScheduler(
        optimizer=optimizer,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16

    history = []
    best_val_mrr = -1.0
    best_path = os.path.join(out_dir, "best_mcq.pt")

    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        seen = 0

        pbar = tqdm(train_loader, desc=f"Train epoch {epoch}/{cfg.epochs}", mininterval=1.0)

        for step, batch in enumerate(pbar, start=1):
            x = batch["x"].to(device, non_blocking=True)
            choice_mask = batch["choice_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=amp_dtype):
                    logits = model(x, choice_mask, mu=mu, sigma=sigma)
                    loss = masked_choice_cross_entropy(
                        logits=logits,
                        labels=labels,
                        choice_mask=choice_mask,
                        label_smoothing=float(cfg.label_smoothing),
                    )
            else:
                logits = model(x, choice_mask, mu=mu, sigma=sigma)
                loss = masked_choice_cross_entropy(
                    logits=logits,
                    labels=labels,
                    choice_mask=choice_mask,
                    label_smoothing=float(cfg.label_smoothing),
                )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            (loss / cfg.grad_accum_steps).backward()

            running_loss += float(loss.item()) * x.size(0)
            seen += int(x.size(0))

            if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

        train_loss = running_loss / max(seen, 1)

        val_metrics = evaluate_ranking_metrics(model, val_loader, device, mu, sigma, k_values=(1, 2, 3, 4))
        test_metrics = evaluate_ranking_metrics(model, test_loader, device, mu, sigma, k_values=(1, 2, 3, 4))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,

            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_precision@1": val_metrics["precision@1"],
            "val_recall@1": val_metrics["recall@1"],
            "val_precision@2": val_metrics["precision@2"],
            "val_recall@2": val_metrics["recall@2"],
            "val_precision@3": val_metrics["precision@3"],
            "val_recall@3": val_metrics["recall@3"],
            "val_precision@4": val_metrics["precision@4"],
            "val_recall@4": val_metrics["recall@4"],
            "val_mrr": val_metrics["mrr"],

            "test_loss": test_metrics["loss"],
            "test_acc": test_metrics["accuracy"],
            "test_precision@1": test_metrics["precision@1"],
            "test_recall@1": test_metrics["recall@1"],
            "test_precision@2": test_metrics["precision@2"],
            "test_recall@2": test_metrics["recall@2"],
            "test_precision@3": test_metrics["precision@3"],
            "test_recall@3": test_metrics["recall@3"],
            "test_precision@4": test_metrics["precision@4"],
            "test_recall@4": test_metrics["recall@4"],
            "test_mrr": test_metrics["mrr"],
        }

        history.append(row)

        print(
            f"[epoch {epoch}] "
            f"train_loss={train_loss:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} "
            f"val_R@2={val_metrics['recall@2']:.4f} "
            f"val_MRR={val_metrics['mrr']:.4f} "
            f"test_acc={test_metrics['accuracy']:.4f} "
            f"test_MRR={test_metrics['mrr']:.4f}"
        )

        # Save only small trainable state. DO NOT save HLCM.
        if val_metrics["mrr"] > best_val_mrr:
            best_val_mrr = float(val_metrics["mrr"])

            small_state = {
                "norm.weight": model.norm.weight.detach().cpu(),
                "norm.bias": model.norm.bias.detach().cpu(),
                "classifier.weight": model.classifier.weight.detach().cpu(),
                "classifier.bias": model.classifier.bias.detach().cpu(),
            }

            torch.save(
                {
                    "cfg": asdict(cfg),
                    "dataset_name": dataset_name,
                    "epoch": epoch,
                    "best_metric": "validation_mrr",
                    "best_val_mrr": best_val_mrr,
                    "best_val_acc": float(val_metrics["accuracy"]),
                    "head_state": small_state,
                    "history": history,
                },
                best_path,
                _use_new_zipfile_serialization=False,
            )

            print(f"[save] best small checkpoint -> {best_path}")

    wall = time.time() - start_time

    # Load best small checkpoint
    best_obj = torch.load(best_path, map_location="cpu")
    head_state = best_obj["head_state"]

    with torch.no_grad():
        model.norm.weight.copy_(head_state["norm.weight"].to(device))
        model.norm.bias.copy_(head_state["norm.bias"].to(device))
        model.classifier.weight.copy_(head_state["classifier.weight"].to(device))
        model.classifier.bias.copy_(head_state["classifier.bias"].to(device))

    final_val = evaluate_ranking_metrics(model, val_loader, device, mu, sigma, k_values=(1, 2, 3, 4))
    final_test = evaluate_ranking_metrics(model, test_loader, device, mu, sigma, k_values=(1, 2, 3, 4))

    summary = {
        "dataset_name": dataset_name,
        "best_checkpoint": best_path,
        "best_epoch": int(best_obj["epoch"]),
        "best_metric": "validation_mrr",

        "final_val": final_val,
        "final_test": final_test,

        "epochs": cfg.epochs,
        "wall_time_sec": float(wall),
        "wall_time_hms": fmt_hms(wall),

        "num_train_examples": len(train_ds),
        "num_val_examples": len(val_ds),
        "num_test_examples": len(test_ds),

        "history": history,
    }

    summary_path = os.path.join(out_dir, "summary_train_eval_ranking.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"[done] {dataset_name}")
    print(json.dumps(summary, indent=2))
    print(f"[saved summary] {summary_path}")

    return summary


# ============================================================
# MAIN
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()

    p.add_argument("--ckpt_path", type=str, default="runs/hyperbolic_cluster/checkpoints/ckpt_best.pt")
    p.add_argument("--normalizer_path", type=str, default="normalizer.pt")
    p.add_argument("--out_dir", type=str, default="runs/mcq_hlcm_openbookqaa")
    p.add_argument("--cache_dir", type=str, default="mcq_cache")

    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--train_batch_size", type=int, default=4)
    p.add_argument("--eval_batch_size", type=int, default=8)
    p.add_argument("--grad_accum_steps", type=int, default=1)

    p.add_argument("--head_lr", type=float, default=3e-5)
    p.add_argument("--weight_decay", type=float, default=0.01)
    p.add_argument("--warmup_ratio", type=float, default=0.06)
    p.add_argument("--max_grad_norm", type=float, default=1.0)

    p.add_argument("--seq_len", type=int, default=8)
    p.add_argument("--chunk_tok_len", type=int, default=256)
    p.add_argument("--encoder_batch_size", type=int, default=64)

    p.add_argument("--prefer_gpu_index", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--label_smoothing", type=float, default=0.10)

    if "ipykernel" in sys.modules:
        return p.parse_args(args=[])

    return p.parse_args()


def main():
    args = parse_args()

    cfg = FinetuneConfig(
        ckpt_path=args.ckpt_path,
        normalizer_path=args.normalizer_path,
        out_dir=args.out_dir,
        cache_dir=args.cache_dir,
        epochs=args.epochs,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        grad_accum_steps=args.grad_accum_steps,
        head_lr=args.head_lr,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        max_grad_norm=args.max_grad_norm,
        seq_len=args.seq_len,
        chunk_tok_len=args.chunk_tok_len,
        encoder_batch_size=args.encoder_batch_size,
        prefer_gpu_index=args.prefer_gpu_index,
        seed=args.seed,
        label_smoothing=args.label_smoothing,
    )

    ensure_dir(cfg.out_dir)

    summary = train_one_dataset(cfg)

    all_path = os.path.join(cfg.out_dir, "openbookqa_train_eval_results.json")
    with open(all_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== ALL DONE ====================")
    print(f"Saved results to: {all_path}")


if __name__ == "__main__":
    main()


==================== TRAIN: OpenBookQA ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] loading mcq_cache/OpenBookQA_train_seq8_tok256.pt
[cache] loading mcq_cache/OpenBookQA_validation_seq8_tok256.pt
[cache] loading mcq_cache/OpenBookQA_test_seq8_tok256.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0


Train epoch 1/5: 100%|█████████████████████████████| 1240/1240 [01:43<00:00, 12.03it/s, loss=1.6126]
                                                                                                    

...

Train epoch 4/5: 100%|█████████████████████████████| 1240/1240 [01:30<00:00, 13.64it/s, loss=1.5724]
                                                                                                    

[epoch 4] train_loss=1.5724 val_acc=0.2480 val_R@2=0.5100 val_MRR=0.5195 test_acc=0.2480 test_MRR=0.5175


Train epoch 5/5: 100%|█████████████████████████████| 1240/1240 [01:30<00:00, 13.64it/s, loss=1.5434]
                                                                                                    

[epoch 5] train_loss=1.5434 val_acc=0.2500 val_R@2=0.5080 val_MRR=0.5202 test_acc=0.2520 test_MRR=0.5202


[done] OpenBookQA
{
  "dataset_name": "OpenBookQA",
  "best_checkpoint": "runs/mcq_hlcm_openbookqaa/OpenBookQA/best_mcq.pt",
  "best_epoch": 1,
  "best_metric": "validation_mrr",
  "final_val": {
    "loss": 1.3861392822265626,
    "accuracy": 0.252,
    "precision@1": 0.252,
    "recall@1": 0.252,
    "precision@2": 0.256,
    "recall@2": 0.512,
    "precision@3": 0.24133333333333207,
    "recall@3": 0.724,
    "precision@4": 0.25,
    "recall@4": 1.0,
    "mrr": 0.5216666666666675
  },
  "final_test": {
    "loss": 1.3876541690826416,
    "accuracy": 0.246,
    "precision@1": 0.246,
    "recall@1": 0.246,
    "precision@2": 0.239,
    "recall@2": 0.478,
    "precision@3": 0.2439999999999987,
    "recall@3": 0.732,
    "precision@4": 0.25,
    "recall@4": 1.0,
    "mrr": 0.513666666666668
  },
  "epochs": 5,
  "wall_time_sec": 510.8457205295563,
  "wall_time_hms": "00:08:30",
  "num_train_examples": 4957,
  "num_val_examples": 500,
  "num_test_examples": 500,
  "history": [
    {
    

In [1]:
#reports accuracy, macro/weighted precision/recall/F1, Precision@k/Recall@k/MRR, Brier score, ECE, and MCE.
# ============================================================
# OpenBookQA EVAL ONLY
# Metrics: Accuracy, Precision/Recall/F1, Precision@k/Recall@k/MRR,
#          Brier score, ECE, MCE
# NO TRAINING
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


@dataclass
class EvalConfig:
    dataset_name: str = "OpenBookQA"
    hf_dataset_name: str = "allenai/openbookqa"
    hf_config_name: str = "main"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_openbookqaa"

    hlcm_ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    mcq_ckpt_path: str = "runs/mcq_hlcm_openbookqaa/OpenBookQA/best_mcq.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 8
    num_workers: int = 0
    head_dropout: float = 0.50

    ece_bins: int = 15
    seed: int = 42
    prefer_gpu_index: int = 0


cfg = EvalConfig()


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if key in alpha:
        idx = alpha[key]
    elif key in numeric:
        idx = numeric[key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(f"answerKey={answer_key} -> idx={idx}, num_choices={num_choices}")
    return idx


def load_normalizer(path: str, device: torch.device):
    if not path or not os.path.exists(path):
        print("[normalizer] not found; continuing without normalizer")
        return None, None
    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded {path}")
    return mu, sigma


def build_amp_dtype(device):
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def encode_text(self, text: str) -> torch.Tensor:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(self.tokenizer.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(chunks) >= self.seq_len:
                break

        out = []
        for i in range(0, len(chunks), self.batch_size):
            out.append(self._embed_chunk_texts(chunks[i:i + self.batch_size]))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(self.seq_len - embs.size(0), self.embed_dim, dtype=embs.dtype)
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []
        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))
        return torch.stack(feats, dim=0)


def load_openbookqa(cfg: EvalConfig):
    try:
        return load_dataset(cfg.hf_dataset_name, cfg.hf_config_name)
    except Exception:
        return load_dataset(cfg.hf_dataset_name)


def normalize_openbookqa(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    stem = str(example["question_stem"])
    choice_texts = list(example["choices"]["text"])
    label = answerkey_to_index(example["answerKey"], len(choice_texts))
    return stem, choice_texts, label


def cache_path(cfg: EvalConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{cfg.dataset_name}_{split_name}_seq{cfg.seq_len}_tok{cfg.chunk_tok_len}.pt",
    )


def build_or_load_cached_split(cfg: EvalConfig, split_name: str, split_data, conceptizer):
    path = cache_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    rows = []
    skipped = 0

    for ex in tqdm(split_data, desc=f"Building cache {split_name}"):
        try:
            stem, choices, label = normalize_openbookqa(ex)
            x = conceptizer.encode_choice_set(stem, choices)
            rows.append({
                "x": x,
                "label": int(label),
                "choice_mask": torch.ones(x.size(0), dtype=torch.bool),
                "num_choices": int(x.size(0)),
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path}; rows={len(rows)}, skipped={skipped}")
    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
        }


def mcq_collate(batch):
    max_c = max(item["x"].size(0) for item in batch)
    b = len(batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)
        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: EvalConfig, device: torch.device):
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.hlcm_ckpt_path):
        raise FileNotFoundError(f"HLCM checkpoint not found: {cfg.hlcm_ckpt_path}")

    obj = torch.load(cfg.hlcm_ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] HLCM loaded from {cfg.hlcm_ckpt_path}")
    print(f"[load] missing={len(missing)}, unexpected={len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model


class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()
        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(self, x, choice_mask, mu=None, sigma=None):
        b, c, t, d = x.shape
        x = x.reshape(b * c, t, d)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)
        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)
        return logits


def load_mcq_checkpoint(model: MCQHead, path: str, device: torch.device):
    if not os.path.exists(path):
        raise FileNotFoundError(f"MCQ checkpoint not found: {path}")

    ckpt = torch.load(path, map_location="cpu")

    if isinstance(ckpt, dict) and "head_state" in ckpt:
        hs = ckpt["head_state"]
        with torch.no_grad():
            model.norm.weight.copy_(hs["norm.weight"].to(device))
            model.norm.bias.copy_(hs["norm.bias"].to(device))
            model.classifier.weight.copy_(hs["classifier.weight"].to(device))
            model.classifier.bias.copy_(hs["classifier.bias"].to(device))
    elif isinstance(ckpt, dict) and "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"], strict=True)
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return ckpt if isinstance(ckpt, dict) else {}


def masked_choice_cross_entropy(logits, labels, choice_mask):
    log_probs = F.log_softmax(logits, dim=-1)
    losses = []

    for i in range(logits.size(0)):
        valid = choice_mask[i]
        valid_idx = torch.nonzero(valid, as_tuple=False).squeeze(-1)
        lp = log_probs[i, valid]

        gold = int(labels[i].item())
        local_pos = (valid_idx == gold).nonzero(as_tuple=False)

        if local_pos.numel() == 0:
            continue

        local_pos = int(local_pos.item())
        losses.append(-lp[local_pos])

    if len(losses) == 0:
        return torch.tensor(0.0, device=logits.device)

    return torch.stack(losses).mean()


def safe_div(a, b):
    return a / b if b else 0.0


def classification_metrics(y_true, y_pred, num_classes: int):
    per_class = {}
    supports = []

    macro_p = macro_r = macro_f1 = 0.0
    weighted_p = weighted_r = weighted_f1 = 0.0

    total = len(y_true)

    for cls in range(num_classes):
        tp = sum(t == cls and p == cls for t, p in zip(y_true, y_pred))
        fp = sum(t != cls and p == cls for t, p in zip(y_true, y_pred))
        fn = sum(t == cls and p != cls for t, p in zip(y_true, y_pred))
        support = sum(t == cls for t in y_true)

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall)

        per_class[f"class_{cls}_precision"] = precision
        per_class[f"class_{cls}_recall"] = recall
        per_class[f"class_{cls}_f1"] = f1
        per_class[f"class_{cls}_support"] = support

        macro_p += precision
        macro_r += recall
        macro_f1 += f1

        weighted_p += precision * support
        weighted_r += recall * support
        weighted_f1 += f1 * support
        supports.append(support)

    return {
        **per_class,
        "macro_precision": macro_p / max(num_classes, 1),
        "macro_recall": macro_r / max(num_classes, 1),
        "macro_f1": macro_f1 / max(num_classes, 1),
        "weighted_precision": weighted_p / max(total, 1),
        "weighted_recall": weighted_r / max(total, 1),
        "weighted_f1": weighted_f1 / max(total, 1),
    }


def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)


@torch.no_grad()
def evaluate_all_metrics(model, loader, device, cfg, mu=None, sigma=None, k_values=(1, 2, 3, 4)):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0
    total_brier = 0.0

    y_true = []
    y_pred = []

    all_confidences = []
    all_correctness = []

    metric_sums = {}
    for k in k_values:
        metric_sums[f"precision@{k}"] = 0.0
        metric_sums[f"recall@{k}"] = 0.0
    metric_sums["mrr"] = 0.0

    max_num_classes = 0

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        loss = masked_choice_cross_entropy(logits, labels, choice_mask)

        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels

        bs = labels.size(0)
        total += bs
        correct += int(batch_correct.sum().item())
        total_loss += float(loss.item()) * bs
        total_brier += float(multiclass_brier_score(probs, labels, choice_mask).sum().item())

        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(preds.detach().cpu().tolist())

        all_confidences.append(probs.max(dim=-1).values.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        max_num_classes = max(max_num_classes, int(choice_mask.sum(dim=1).max().item()))

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(bs):
            gold = int(labels[i].item())
            valid_count = int(choice_mask[i].sum().item())
            ranked_i = ranked[i, :valid_count]

            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)
            if rank_pos.numel() == 0:
                continue

            rank = int(rank_pos.item()) + 1
            metric_sums["mrr"] += 1.0 / rank

            for k in k_values:
                kk = min(k, valid_count)
                hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                metric_sums[f"recall@{k}"] += hit
                metric_sums[f"precision@{k}"] += hit / kk

    confidences = torch.cat(all_confidences) if all_confidences else torch.empty(0)
    correctness = torch.cat(all_correctness) if all_correctness else torch.empty(0)
    ece, mce = expected_calibration_error(confidences, correctness, n_bins=cfg.ece_bins)

    metrics = {
        "num_examples": total,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": cfg.ece_bins,
    }

    for key, value in metric_sums.items():
        metrics[key] = value / max(total, 1)

    metrics.update(classification_metrics(y_true, y_pred, num_classes=max_num_classes))
    return metrics


def main():
    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("EVAL ONLY. No training.")
    print("MCQ checkpoint:", cfg.mcq_ckpt_path)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    raw_ds = load_openbookqa(cfg)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=device,
    )

    val_rows = build_or_load_cached_split(cfg, "validation", raw_ds["validation"], conceptizer)
    test_rows = build_or_load_cached_split(cfg, "test", raw_ds["test"], conceptizer)

    del conceptizer
    cuda_cleanup()

    val_loader = DataLoader(
        MCQFeatureDataset(val_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    test_loader = DataLoader(
        MCQFeatureDataset(test_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)
    ckpt_info = load_mcq_checkpoint(model, cfg.mcq_ckpt_path, device)

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    val_metrics = evaluate_all_metrics(model, val_loader, device, cfg, mu, sigma, k_values=(1, 2, 3, 4))
    test_metrics = evaluate_all_metrics(model, test_loader, device, cfg, mu, sigma, k_values=(1, 2, 3, 4))

    summary = {
        "dataset_name": cfg.dataset_name,
        "checkpoint": cfg.mcq_ckpt_path,
        "checkpoint_info": {
            "epoch": ckpt_info.get("epoch", None),
            "best_metric": ckpt_info.get("best_metric", None),
            "best_val_mrr": ckpt_info.get("best_val_mrr", None),
            "best_val_acc": ckpt_info.get("best_val_acc", None),
        },
        "validation": val_metrics,
        "test": test_metrics,
    }

    save_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(save_dir)

    save_path = os.path.join(save_dir, "eval_precision_recall_f1_mrr_brier_ece.json")
    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))
    print(f"Saved metrics to: {save_path}")

    del model
    cuda_cleanup()


if __name__ == "__main__":
    main()

Device: cuda:0
EVAL ONLY. No training.
MCQ checkpoint: runs/mcq_hlcm_openbookqaa/OpenBookQA/best_mcq.pt


Using the latest cached version of the dataset since allenai/openbookqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/user/.cache/huggingface/datasets/allenai___openbookqa/main/0.0.0/388097ea7776314e93a529163e0fea805b8a6454 (last modified on Wed Mar 18 08:37:00 2026).


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading mcq_cache/OpenBookQA_validation_seq8_tok256.pt
[cache] loading mcq_cache/OpenBookQA_test_seq8_tok256.pt
[load] HLCM loaded from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing=0, unexpected=0
[normalizer] loaded normalizer.pt


{
  "dataset_name": "OpenBookQA",
  "checkpoint": "runs/mcq_hlcm_openbookqaa/OpenBookQA/best_mcq.pt",
  "checkpoint_info": {
    "epoch": 1,
    "best_metric": "validation_mrr",
    "best_val_mrr": 0.5216666666666675,
    "best_val_acc": 0.252
  },
  "validation": {
    "num_examples": 500,
    "loss": 1.3861392345428467,
    "accuracy": 0.252,
    "brier_score": 0.7499150104522705,
    "ece": 0.012705066469024917,
    "mce": 0.18635734915733337,
    "ece_bins": 15,
    "precision@1": 0.252,
    "recall@1": 0.252,
    "precision@2": 0.256,
    "recall@2": 0.512,
    "precision@3": 0.24133333333333207,
    "recall@3": 0.724,
    "precision@4": 0.25,
    "recall@4": 1.0,
    "mrr": 0.5216666666666675,
    "class_0_precision": 0.2446043165467626,
    "class_0_recall": 0.2698412698412698,
    "class_0_f1": 0.2566037735849057,
    "class_0_support": 126,
    "class_1_precision": 0.26356589147286824,
    "class_1_recall": 0.2463768115942029,
    "class_1_f1": 0.2546816479400749,
    "class_1

In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "OpenBookQA"

    cache_dir: str = "mcq_cache"
    out_dir: str = "runs/mcq_hlcm_openbookqa"

    best_ckpt_path: str = "runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 64

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    head_dropout: float = 0.50

    eval_batch_size: int = 8
    num_workers: int = 4
    prefer_gpu_index: int = 0
    seed: int = 42

    split: str = "test"

    # If True, includes DeBERTa conceptization time when cache is missing.
    # If cache exists, inference time measures only H-LCM + MCQ head forward pass.
    build_cache_if_missing: bool = True


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = max(float(seconds), 0.0)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def answerkey_to_index(answer_key: str, num_choices: int) -> int:
    answer_key = str(answer_key).strip()
    alpha = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    numeric = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4}

    if answer_key in alpha:
        idx = alpha[answer_key]
    elif answer_key in numeric:
        idx = numeric[answer_key]
    else:
        raise ValueError(f"Unknown answerKey: {answer_key}")

    if idx >= num_choices:
        raise ValueError(
            f"answerKey={answer_key} -> idx={idx}, but num_choices={num_choices}"
        )

    return idx


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.encoder = AutoModel.from_pretrained(model_name).to(device)
        self.encoder.eval()

        for p in self.encoder.parameters():
            p.requires_grad = False

        self.embed_dim = int(self.encoder.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.embed_dim, dtype=torch.float32)

        inputs = self.tokenizer(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device, non_blocking=True) for k, v in inputs.items()}

        if self.device.type == "cuda":
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            with amp.autocast(device_type="cuda", enabled=True, dtype=dtype):
                out = self.encoder(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.encoder(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _text_to_chunk_texts(self, text: str) -> List[str]:
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]

        if len(ids) == 0:
            return []

        chunk_texts = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunk_ids = ids[i:i + self.chunk_tok_len]
            chunk_text = self.tokenizer.decode(
                chunk_ids,
                clean_up_tokenization_spaces=True,
            )
            chunk_texts.append(chunk_text)

        return chunk_texts

    def encode_text(self, text: str) -> torch.Tensor:
        chunk_texts = self._text_to_chunk_texts(text)

        if len(chunk_texts) == 0:
            return torch.zeros(self.seq_len, self.embed_dim, dtype=torch.float32)

        chunk_texts = chunk_texts[:self.seq_len]

        out = []
        for i in range(0, len(chunk_texts), self.batch_size):
            batch = chunk_texts[i:i + self.batch_size]
            out.append(self._embed_chunk_texts(batch))

        embs = torch.cat(out, dim=0)

        if embs.size(0) < self.seq_len:
            pad = torch.zeros(
                self.seq_len - embs.size(0),
                embs.size(1),
                dtype=embs.dtype,
            )
            embs = torch.cat([embs, pad], dim=0)

        return embs[:self.seq_len]

    def encode_choice_set(self, question_stem: str, choice_texts: List[str]) -> torch.Tensor:
        feats = []

        for ch in choice_texts:
            text = f"Question: {question_stem}\nAnswer Choice: {ch}"
            feats.append(self.encode_text(text))

        return torch.stack(feats, dim=0)


# ============================================================
# DATA
# ============================================================

def load_raw_mcq_dataset(dataset_name: str):
    if dataset_name == "OpenBookQA":
        return load_dataset("allenai/openbookqa")

    raise ValueError(f"Unknown dataset: {dataset_name}")


def normalize_choices_field(example: Dict[str, Any]) -> Tuple[str, List[str], int]:
    if "question_stem" not in example:
        raise KeyError("question_stem")

    stem = str(example["question_stem"])

    if "choices" not in example:
        raise KeyError("choices")

    choices = example["choices"]

    if not isinstance(choices, dict):
        raise TypeError(f"Expected choices dict, got {type(choices)}")

    if "text" not in choices:
        raise KeyError("choices.text")

    choice_texts = [str(x) for x in choices["text"]]

    if "answerKey" not in example:
        raise KeyError("answerKey")

    label = answerkey_to_index(example["answerKey"], len(choice_texts))

    return stem, choice_texts, label


def cache_path_for_split(cfg: InferenceConfig, split_name: str) -> str:
    safe_ds = cfg.dataset_name.replace("/", "_")
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_seq{cfg.seq_len}_tok{cfg.chunk_tok_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    split_data,
    conceptizer: Optional[DebertaConceptizer],
) -> List[Dict[str, Any]]:
    ensure_dir(cfg.cache_dir)

    path = cache_path_for_split(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(
            f"Cache file not found: {path}\n"
            f"Set build_cache_if_missing=True if you want to build it."
        )

    if conceptizer is None:
        raise RuntimeError("conceptizer is required to build cache")

    rows = []
    skipped = 0

    print(f"[cache] building {cfg.dataset_name} / {split_name}")

    for ex in tqdm(split_data, desc=f"Conceptizing {cfg.dataset_name}-{split_name}"):
        try:
            stem, choice_texts, label = normalize_choices_field(ex)
            x = conceptizer.encode_choice_set(stem, choice_texts)
            choice_mask = torch.ones(x.size(0), dtype=torch.bool)

            rows.append(
                {
                    "x": x,
                    "label": int(label),
                    "choice_mask": choice_mask,
                    "num_choices": int(x.size(0)),
                }
            )
        except Exception as e:
            skipped += 1
            print(
                f"[warn] skipped one example in {cfg.dataset_name}-{split_name}: "
                f"{type(e).__name__}: {e}"
            )

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class MCQFeatureDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "choice_mask": r["choice_mask"],
            "num_choices": r["num_choices"],
        }


def mcq_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    max_c = max(item["x"].size(0) for item in batch)
    t = batch[0]["x"].size(1)
    d = batch[0]["x"].size(2)
    b = len(batch)

    x = torch.zeros(b, max_c, t, d, dtype=torch.float32)
    choice_mask = torch.zeros(b, max_c, dtype=torch.bool)
    labels = torch.zeros(b, dtype=torch.long)

    for i, item in enumerate(batch):
        c = item["x"].size(0)

        x[i, :c] = item["x"]
        choice_mask[i, :c] = item["choice_mask"]
        labels[i] = item["label"]

    return {
        "x": x,
        "choice_mask": choice_mask,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class MCQHead(nn.Module):
    def __init__(self, model: HyperbolicLCM, dropout: float = 0.5):
        super().__init__()

        self.model = model
        dim = model.layers[0].attn.embed_dim

        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(dim, 1)

    def forward(
        self,
        x: torch.Tensor,
        choice_mask: torch.Tensor,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, c, t, din = x.shape
        x = x.reshape(b * c, t, din)

        if mu is not None and sigma is not None:
            x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

        with torch.no_grad():
            h = self.model(x)

        last_h = h[:, -1, :]
        last_tan = self.model.manifold.logmap0(last_h)

        last_tan = self.norm(last_tan)
        last_tan = self.dropout(last_tan)

        logits = self.classifier(last_tan).view(b, c)
        logits = logits.masked_fill(~choice_mask, -1e9)

        return logits


def build_hlcm_from_cfg(cfg: InferenceConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_inference_model(cfg: InferenceConfig, device: torch.device) -> MCQHead:
    hlcm = build_hlcm_from_cfg(cfg).to(device)
    model = MCQHead(hlcm, dropout=cfg.head_dropout).to(device)

    if os.path.exists(cfg.best_ckpt_path):
        print(f"[load] loading fine-tuned checkpoint: {cfg.best_ckpt_path}")

        obj = torch.load(cfg.best_ckpt_path, map_location=device)

        if "model_state" not in obj:
            raise KeyError("Checkpoint does not contain 'model_state'.")

        missing, unexpected = model.load_state_dict(obj["model_state"], strict=False)

        print(f"[load] missing keys: {len(missing)}")
        print(f"[load] unexpected keys: {len(unexpected)}")
        print("[load] fine-tuned MCQ checkpoint loaded")

    else:
        print("=" * 80)
        print("[warning] Fine-tuned best_mcq.pt was NOT found.")
        print("[warning] Running inference with randomly initialized MCQ head.")
        print("[warning] Inference time is valid, but accuracy is NOT meaningful.")
        print("=" * 80)

        # Optional: load only pretrained HLCM backbone if available
        pretrained_path = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"

        if os.path.exists(pretrained_path):
            obj = torch.load(pretrained_path, map_location="cpu")
            state = obj["model"] if "model" in obj else obj

            missing, unexpected = hlcm.load_state_dict(state, strict=False)

            print(f"[load] loaded pretrained HLCM backbone from {pretrained_path}")
            print(f"[load] HLCM missing keys: {len(missing)}")
            print(f"[load] HLCM unexpected keys: {len(unexpected)}")
        else:
            print("[warning] pretrained HLCM checkpoint also not found.")
            print("[warning] using fully random HLCM + random MCQ head.")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model


# ============================================================
# INFERENCE + TIMING
# ============================================================

@torch.no_grad()
def run_inference_with_time(
    model: MCQHead,
    loader: DataLoader,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    save_path: Optional[str] = None,
) -> Dict[str, Any]:
    model.eval()

    total = 0
    correct = 0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    for batch in tqdm(loader, desc="Inference"):
        x = batch["x"].to(device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(x, choice_mask, mu=mu, sigma=sigma)
        preds = logits.argmax(dim=-1)

        batch_correct = preds.eq(labels)

        for i in range(x.size(0)):
            predictions.append(
                {
                    "example_index": total + i,
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(batch_correct[i].item()),
                    "logits": [float(v) for v in logits[i].detach().cpu().tolist()],
                }
            )

        correct += int(batch_correct.sum().item())
        total += int(x.size(0))

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - start_time

    results = {
        "num_examples": total,
        "correct": correct,
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

        print(f"[save] inference results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_openbookqa(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print(f"[device] {device}")

    ensure_dir(cfg.cache_dir)

    dataset_out_dir = os.path.join(cfg.out_dir, cfg.dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)

    raw_ds = load_raw_mcq_dataset(cfg.dataset_name)

    conceptizer = None
    cache_path = cache_path_for_split(cfg, cfg.split)

    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if cfg.build_cache_if_missing:
            conceptizer = DebertaConceptizer(
                model_name=cfg.encoder_name,
                chunk_tok_len=cfg.chunk_tok_len,
                seq_len=cfg.seq_len,
                batch_size=cfg.encoder_batch_size,
                device=device,
            )

            if device.type == "cuda":
                torch.cuda.synchronize()

            cache_start = time.perf_counter()

            rows = build_or_load_cached_split(
                cfg=cfg,
                split_name=cfg.split,
                split_data=raw_ds[cfg.split],
                conceptizer=conceptizer,
            )

            if device.type == "cuda":
                torch.cuda.synchronize()

            cache_build_time_sec = time.perf_counter() - cache_start
        else:
            raise FileNotFoundError(cache_path)
    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=cfg.split,
            split_data=raw_ds[cfg.split],
            conceptizer=None,
        )

    print(f"[data] split={cfg.split}, examples={len(rows)}")

    ds = MCQFeatureDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=mcq_collate,
        drop_last=False,
    )

    model = load_inference_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    save_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_results.json",
    )

    inference_results = run_inference_with_time(
        model=model,
        loader=loader,
        device=device,
        mu=mu,
        sigma=sigma,
        save_path=save_path,
    )

    summary = {
        "dataset_name": cfg.dataset_name,
        "split": cfg.split,
        "checkpoint": cfg.best_ckpt_path,
        "cache_file": cache_path,
        "num_examples": inference_results["num_examples"],
        "correct": inference_results["correct"],
        "accuracy": inference_results["accuracy"],
        "accuracy_percent": inference_results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": inference_results["inference_time_sec"],
        "inference_time_hms": inference_results["inference_time_hms"],
        "time_per_example_sec": inference_results["time_per_example_sec"],
        "examples_per_second": inference_results["examples_per_second"],
    }

    summary_path = os.path.join(
        dataset_out_dir,
        f"inference_only_{cfg.split}_summary.json",
    )

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, inference_results

In [2]:
cfg = InferenceConfig(
    dataset_name="OpenBookQA",

    best_ckpt_path="runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt",
    normalizer_path="normalizer.pt",

    cache_dir="mcq_cache",
    out_dir="runs/mcq_hlcm_openbookqa",

    split="test",
    eval_batch_size=8,
    prefer_gpu_index=0,

    build_cache_if_missing=True,
)

summary, inference_results = inference_only_openbookqa(cfg)

[device] cuda:0


Using the latest cached version of the dataset since allenai/openbookqa couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'main' at /home/user/.cache/huggingface/datasets/allenai___openbookqa/main/0.0.0/388097ea7776314e93a529163e0fea805b8a6454 (last modified on Wed Mar 18 08:37:00 2026).


[cache] loading mcq_cache/OpenBookQA_test_seq8_tok256.pt
[data] split=test, examples=500
[warning] Fine-tuned best_mcq.pt was NOT found.
[warning] Running inference with randomly initialized MCQ head.
[warning] Inference time is valid, but accuracy is NOT meaningful.
[load] loaded pretrained HLCM backbone from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] HLCM missing keys: 0
[load] HLCM unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference: 100%|████████████████████████████████████████████████████| 63/63 [00:11<00:00,  5.63it/s]

[save] inference results -> runs/mcq_hlcm_openbookqa/OpenBookQA/inference_only_test_results.json

==================== INFERENCE DONE ====================
{
  "dataset_name": "OpenBookQA",
  "split": "test",
  "checkpoint": "runs/mcq_hlcm_openbookqa/OpenBookQA/best_mcq.pt",
  "cache_file": "mcq_cache/OpenBookQA_test_seq8_tok256.pt",
  "num_examples": 500,
  "correct": 130,
  "accuracy": 0.26,
  "accuracy_percent": 26.0,
  "cache_build_time_sec": 0.0,
  "cache_build_time_hms": "00:00:00",
  "inference_time_sec": 11.271639277692884,
  "inference_time_hms": "00:00:11",
  "time_per_example_sec": 0.02254327855538577,
  "examples_per_second": 44.35912006069286
}
[summary saved] runs/mcq_hlcm_openbookqa/OpenBookQA/inference_only_test_summary.json
